# Import libraries

In [ ]:
import sys
from os import listdir
from os.path import isfile, join
from official_libraries import *
from utils import *
import spm1d
from scipy.signal import find_peaks, argrelextrema
import numpy as np
from tslearn.clustering import TimeSeriesKMeans
from tslearn.utils import to_time_series_dataset
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib qt

# Load preprocessed data

In [ ]:
mypath = '.../detection_comVT/'
onlyfiles = [f for f in listdir(mypath) if isfile(join(mypath, f))]

t1_files = [i for i in onlyfiles if i.startswith('1_')]
t2_files = [i for i in onlyfiles if i.startswith('2_')]
subjs = ['1','10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

t1 = {}
for p, s in zip(t1_files, subjs):
    with open(mypath + p, 'rb') as f:
        t1['{}'.format(s)] = pickle.load(f)

t2 = {}
for p, s in zip(t2_files, subjs):
    with open(mypath + p, 'rb') as f:
        t2['{}'.format(s)] = pickle.load(f)

mypath = '/Volumes/Seagate/biomech_analysis/clustering/curves/'
onlyfiles = [f for f in listdir(mypath) if isfile(join(mypath, f))]

t1_curves_files = [i for i in onlyfiles if i.startswith('1_')]
t2_curves_files = [i for i in onlyfiles if i.startswith('2_')]
subjs = ['1','10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

t1_curves = {}
for p, s in zip(t1_curves_files, subjs):
    with open(mypath + p, 'rb') as f:
        t1_curves['{}'.format(s)] = pickle.load(f)

t2_curves = {}
for p, s in zip(t2_curves_files, subjs):
    with open(mypath + p, 'rb') as f:
        t2_curves['{}'.format(s)] = pickle.load(f)

t1_curves['1']['weight'], t2_curves['1']['weight'] = 67, 67
t1_curves['2']['weight'], t2_curves['2']['weight'] = 70, 70
t1_curves['3']['weight'], t2_curves['3']['weight'] = 70, 70
t1_curves['4']['weight'], t2_curves['4']['weight'] = 63, 63
t1_curves['5']['weight'], t2_curves['5']['weight'] = 75, 75
t1_curves['6']['weight'], t2_curves['6']['weight'] = 72, 72
t1_curves['7']['weight'], t2_curves['7']['weight'] = 70, 70
t1_curves['8']['weight'], t2_curves['8']['weight'] = 68, 68
t1_curves['9']['weight'], t2_curves['9']['weight'] = 55, 55
t1_curves['10']['weight'], t2_curves['10']['weight'] = 63, 63
t1_curves['11']['weight'], t2_curves['11']['weight'] = 90, 90
t1_curves['12']['weight'], t2_curves['12']['weight'] = 57, 57

# Curve registration

## Unimanual

In [8]:
subj_1=['1','1', '2','2','2','2','2','3','3','3','4','4','4','4','4','5','5','5','5','5','6','6','6','6','6',
                            '7','7','7','7','7','8','8','8','8','8','9','9','9','9','9', '10','10','11','11','11','11','11', '12']

res_com_1 = pd.DataFrame({
    '1_1': interpolation(resample_fixed(t1_curves['1']['1']['com_vt'], 100)[0]).values, '1_2': interpolation(resample_fixed(t1_curves['1']['2']['com_vt'], 100)[0]).values,
    '2_1': interpolation(resample_fixed(t1_curves['2']['1']['com_vt'], 100)[0]).values, '2_2': interpolation(resample_fixed(t1_curves['2']['2']['com_vt'], 100)[0]).values,                    
    '2_3': interpolation(resample_fixed(t1_curves['2']['3']['com_vt'], 100)[0]).values, '2_4': interpolation(resample_fixed(t1_curves['2']['4']['com_vt'], 100)[0]).values,                    
    '2_5': interpolation(resample_fixed(t1_curves['2']['5']['com_vt'], 100)[0]).values, 
    '3_1': interpolation(resample_fixed(t1_curves['3']['1']['com_vt'], 100)[0]).values, '3_2': interpolation(resample_fixed(t1_curves['3']['2']['com_vt'], 100)[0]).values,                    
    '4_1': interpolation(resample_fixed(t1_curves['4']['1']['com_vt'], 100)[0]).values, '4_2': interpolation(resample_fixed(t1_curves['4']['2']['com_vt'], 100)[0]).values, 
    '4_3': interpolation(resample_fixed(t1_curves['4']['3']['com_vt'], 100)[0]).values, '4_4': interpolation(resample_fixed(t1_curves['4']['4']['com_vt'], 100)[0]).values,                    
    '4_5': interpolation(resample_fixed(t1_curves['4']['5']['com_vt'], 100)[0]).values,                    
    '5_1': interpolation(resample_fixed(t1_curves['5']['1']['com_vt'], 100)[0]).values, '5_2': interpolation(resample_fixed(t1_curves['5']['2']['com_vt'], 100)[0]).values,                    
    '5_3': interpolation(resample_fixed(t1_curves['5']['3']['com_vt'], 100)[0]).values, '5_4': interpolation(resample_fixed(t1_curves['5']['4']['com_vt'], 100)[0]).values,                    
    '5_5': interpolation(resample_fixed(t1_curves['5']['5']['com_vt'], 100)[0]).values,                    
    '6_1': interpolation(resample_fixed(t1_curves['6']['1']['com_vt'], 100)[0]).values, '6_2': interpolation(resample_fixed(t1_curves['6']['2']['com_vt'], 100)[0]).values,                    
    '6_3': interpolation(resample_fixed(t1_curves['6']['3']['com_vt'], 100)[0]).values, '6_4': interpolation(resample_fixed(t1_curves['6']['4']['com_vt'], 100)[0]).values,                    
    '6_5': interpolation(resample_fixed(t1_curves['6']['5']['com_vt'], 100)[0]).values,                    
    '7_1': interpolation(resample_fixed(t1_curves['7']['1']['com_vt'], 100)[0]).values, '7_2': interpolation(resample_fixed(t1_curves['7']['2']['com_vt'], 100)[0]).values,                    
    '7_3': interpolation(resample_fixed(t1_curves['7']['3']['com_vt'], 100)[0]).values, '7_4': interpolation(resample_fixed(t1_curves['7']['4']['com_vt'], 100)[0]).values,                    
    '7_5': interpolation(resample_fixed(t1_curves['7']['5']['com_vt'], 100)[0]).values,                   
    '8_1': interpolation(resample_fixed(t1_curves['8']['1']['com_vt'], 100)[0]).values, '8_2': interpolation(resample_fixed(t1_curves['8']['2']['com_vt'], 100)[0]).values,                    
    '8_3': interpolation(resample_fixed(t1_curves['8']['3']['com_vt'], 100)[0]).values, '8_4': interpolation(resample_fixed(t1_curves['8']['4']['com_vt'], 100)[0]).values,                    
    '8_5': interpolation(resample_fixed(t1_curves['8']['5']['com_vt'], 100)[0]).values,                    
    '9_1': interpolation(resample_fixed(t1_curves['9']['1']['com_vt'], 100)[0]).values, '9_2': interpolation(resample_fixed(t1_curves['9']['2']['com_vt'], 100)[0]).values,                   
    '9_3': interpolation(resample_fixed(t1_curves['9']['3']['com_vt'], 100)[0]).values, '9_4': interpolation(resample_fixed(t1_curves['9']['4']['com_vt'], 100)[0]).values,                   
    '9_5': interpolation(resample_fixed(t1_curves['9']['5']['com_vt'], 100)[0]).values,                   
    '10_1': interpolation(resample_fixed(t1_curves['10']['1']['com_vt'], 100)[0]).values, '10_2': interpolation(resample_fixed(t1_curves['10']['2']['com_vt'], 100)[0]).values, 
    '11_1': interpolation(resample_fixed(t1_curves['11']['1']['com_vt'], 100)[0]).values, '11_2': interpolation(resample_fixed(t1_curves['11']['2']['com_vt'], 100)[0]).values, 
    '11_3': interpolation(resample_fixed(t1_curves['11']['3']['com_vt'], 100)[0]).values, '11_4': interpolation(resample_fixed(t1_curves['11']['4']['com_vt'], 100)[0]).values, 
    '11_5': interpolation(resample_fixed(t1_curves['11']['5']['com_vt'], 100)[0]).values, 
    '12_1': interpolation(resample_fixed(t1_curves['12']['1']['com_vt'], 100)[0]).values
    })

com_vt = res_com_1.mean(numeric_only=True, axis=1)
com_vt_std = res_com_1.std(numeric_only=True, axis=1)
com_vt_diff = com_vt.diff() * 200
com_vt_diff_abs = abs(com_vt_diff)

peaks, val = find_peaks(com_vt_diff_abs, height=com_vt_diff_abs.max()/2) 

plt.figure()
plt.plot(com_vt_diff)
plt.plot(peaks, com_vt_diff[peaks], "x")
plt.plot(np.zeros_like(com_vt_diff), "--", color="gray")

local_minima = argrelextrema(np.array(com_vt_diff_abs), np.less)[0]
plt.plot(local_minima, com_vt_diff[local_minima], 'x', label='peaks')
events = np.array([1, local_minima[0], 99])
plt.plot(events, com_vt_diff[events], 'x', label='peaks')
events_1 = np.array([0, local_minima[0], 100])
curve_registration_params_1 = {'res_com_mean': com_vt,'res_com_std': com_vt_std,'events': events_1}
# with open('/Volumes/Seagate/biomech_analysis/clustering/'+ 'curve_registration_params_1.pkl', 'wb') as handle:
#     pickle.dump(curve_registration_params, handle, protocol=pickle.HIGHEST_PROTOCOL) 

# with open('/Volumes/Seagate/biomech_analysis/clustering/'+ 'curve_registration_params_1.pkl', 'rb') as f:
#     params_1 = pickle.load(f)
params_1 = curve_registration_params_1
com_vt_registered_1, com_ap_registered_1, cop_ap_registered_1, cop_ml_registered_1 = {}, {}, {}, {}
com_vt_vel_registered_1, com_ap_vel_registered_1, cop_ap_vel_registered_1, cop_ml_vel_registered_1 = {}, {}, {}, {}
trunk_registered_1, knee_registered_1 = {}, {}
fx_1, fy_1, fz_1 = {}, {}, {}
rmalx_registered_1, lmalx_registered_1, rtoez_registered_1, ltoez_registered_1 = {}, {}, {}, {}
rheelx_registered_1, rheely_registered_1, rheelz_registered_1 = {}, {}, {}
lheelx_registered_1, lheely_registered_1, lheelz_registered_1 = {}, {}, {}

subjs = ['1', '2', '3', '4', '5', '6', '7', '8', '9','10', '11', '12']
middle = []
for i in subjs:
    for j in t1_curves[i]:
        if j != 'points' and j!='whole_trunk' and j!='weight':

            com_vt_interp = interpolation(t1_curves[i][j]['com_vt']) / t1[i]['d']
            com_ap_interp = interpolation(t1_curves[i][j]['com_ap']) / t1[i]['d']
            cop_ap_interp = interpolation(t1_curves[i][j]['cop_ap']) / t1[i]['d']
            cop_ml_interp = interpolation(t1_curves[i][j]['cop_ml']) / t1[i]['d']
            trunk_interp = interpolation(t1_curves[i][j]['trunk'])
            fx = interpolation(t1_curves[i][j]['fx']) / t1_curves[i]['weight']
            fy = interpolation(t1_curves[i][j]['fy']) / t1_curves[i]['weight']
            fz = interpolation(t1_curves[i][j]['fz']) / t1_curves[i]['weight']

            b, a = signal.butter(4, 20, 'lowpass', fs=200)
            fx_filt = signal.filtfilt(b, a, fx)
            fy_filt = signal.filtfilt(b, a, fy)
            fz_filt = signal.filtfilt(b, a, fz)

            com_vt_interp_vel = com_vt_interp.diff() / 200
            com_ap_interp_vel = com_ap_interp.diff() / 200
            cop_ap_interp_vel = cop_ap_interp.diff() / 200
            cop_ml_interp_vel = cop_ml_interp.diff() / 200

            # b, a = signal.butter(3, 20, 'lowpass', fs=200)
            # cop_ap_interp_vel = signal.filtfilt(b, a, cop_ap_interp_vel_)
            # cop_ml_interp_vel = signal.filtfilt(b, a, cop_ml_interp_vel_)

            rmalx_interp = interpolation(t1_curves[i][j]['rmal'][:,0])
            lmalx_interp = interpolation(t1_curves[i][j]['lmal'][:,0])
            rtoez_interp = interpolation(t1_curves[i][j]['rtoe'][:,2])
            ltoez_interp = interpolation(t1_curves[i][j]['ltoe'][:,2])
            rheelx_interp = interpolation(t1_curves[i][j]['rheel'][:,0])
            rheely_interp = interpolation(t1_curves[i][j]['rheel'][:,1])
            rheelz_interp = interpolation(t1_curves[i][j]['rheel'][:,2])
            lheelx_interp = interpolation(t1_curves[i][j]['lheel'][:,0])
            lheely_interp = interpolation(t1_curves[i][j]['lheel'][:,1])
            lheelz_interp = interpolation(t1_curves[i][j]['lheel'][:,2])

            if i == '9':
                knee_interp = interpolation(t1_curves[i][j]['knee_left'])
                knee_interp = interpolation(t1_curves[i][j]['knee_left'])
            else:
                knee_interp = interpolation(t1_curves[i][j]['knee_right'])
                knee_interp = interpolation(t1_curves[i][j]['knee_right'])

            com_vt_diff = np.diff(com_vt_interp) * 200
            com_vt_diff_abs = abs(com_vt_diff)
            peaks, val = find_peaks(com_vt_diff_abs, height=com_vt_diff_abs.max()/2) 
            local_minima = argrelextrema(np.array(com_vt_diff_abs), np.less)[0]
            events = [1, local_minima[0], len(com_vt_diff)]
            middle.append(local_minima[0])

            # print(events)
            com_vt_vel_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(com_vt_interp_vel),events, params_1['events'])
            com_ap_vel_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(com_ap_interp_vel),events, params_1['events'])
            cop_ap_vel_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(cop_ap_interp_vel),events, params_1['events'])
            cop_ml_vel_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(cop_ml_interp_vel),events, params_1['events'])
            com_vt_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(com_vt_interp),events, params_1['events'])
            com_ap_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(com_ap_interp),events, params_1['events'])
            cop_ap_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(cop_ap_interp),events, params_1['events'])
            cop_ml_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(cop_ml_interp),events, params_1['events'])
            trunk_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(trunk_interp),events, params_1['events'])
            knee_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(knee_interp),events, params_1['events'])
            fx_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(fx_filt),events, params_1['events'])
            fy_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(fy_filt),events, params_1['events'])
            fz_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(fz_filt),events, params_1['events'])
            rmalx_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(rmalx_interp),events, params_1['events'])
            lmalx_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(lmalx_interp),events, params_1['events'])
            rtoez_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(rtoez_interp),events, params_1['events'])
            ltoez_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(ltoez_interp),events, params_1['events'])
            rheelx_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(rheelx_interp),events, params_1['events'])
            rheely_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(rheely_interp),events, params_1['events'])
            rheelz_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(rheelz_interp),events, params_1['events'])
            lheelx_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(lheelx_interp),events, params_1['events'])
            lheely_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(lheely_interp),events, params_1['events'])
            lheelz_registered_1['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(lheelz_interp),events, params_1['events'])

com_vt_registered_1, com_ap_registered_1 = pd.DataFrame(com_vt_registered_1), pd.DataFrame(com_ap_registered_1)
cop_ap_registered_1 = pd.DataFrame(cop_ap_registered_1)
cop_ml_registered_1 = pd.DataFrame(cop_ml_registered_1)
trunk_registered_1, knee_registered_1 = pd.DataFrame(trunk_registered_1), pd.DataFrame(knee_registered_1)
fx_1, fy_1,fz_1  = pd.DataFrame(fx_1), pd.DataFrame(fy_1), pd.DataFrame(fz_1)

com_vt_vel_registered_1, com_ap_vel_registered_1 = pd.DataFrame(com_vt_vel_registered_1), pd.DataFrame(com_ap_vel_registered_1)
cop_ap_vel_registered_1 = pd.DataFrame(cop_ap_vel_registered_1)
cop_ml_vel_registered_1 = pd.DataFrame(cop_ml_vel_registered_1)

com_vt_registered_1 -= com_vt_registered_1.iloc[0]
com_ap_registered_1 -= com_ap_registered_1.iloc[0]
cop_ap_registered_1 -= cop_ap_registered_1.iloc[0]
cop_ml_registered_1 -= cop_ml_registered_1.iloc[0]
trunk_registered_1 -= trunk_registered_1.iloc[0]
knee_registered_1 -= knee_registered_1.iloc[0]

rmalx_registered_1 = pd.DataFrame(rmalx_registered_1)
lmalx_registered_1 = pd.DataFrame(lmalx_registered_1)
rtoez_registered_1 = pd.DataFrame(rtoez_registered_1)
ltoez_registered_1 = pd.DataFrame(ltoez_registered_1)
rheelx_registered_1 = pd.DataFrame(rheelx_registered_1)
rheely_registered_1 = pd.DataFrame(rheely_registered_1)
rheelz_registered_1 = pd.DataFrame(rheelz_registered_1)
lheelx_registered_1 = pd.DataFrame(lheelx_registered_1)
lheely_registered_1 = pd.DataFrame(lheely_registered_1)
lheelz_registered_1 = pd.DataFrame(lheelz_registered_1)

## Bimanual

In [9]:
subj_2=['1','1', '1', '2','2','2','2','2','3','3','4','4','4','4','4', '4','5','5','5','5','6','6','6','6','6',
     '7','7','7','7','7','8','8','8','8','8','9','9','9','9','9', '10','10','10','10','11','11','11','11','11','12','12','12']

res_com_2 = pd.DataFrame({
    '1_1': interpolation(resample_fixed(t2_curves['1']['1']['com_vt'], 100)[0]).values, '1_2': interpolation(resample_fixed(t2_curves['1']['2']['com_vt'], 100)[0]).values,
    '2_1': interpolation(resample_fixed(t2_curves['2']['1']['com_vt'], 100)[0]).values, '2_2': interpolation(resample_fixed(t2_curves['2']['2']['com_vt'], 100)[0]).values,                    
    '2_3': interpolation(resample_fixed(t2_curves['2']['3']['com_vt'], 100)[0]).values, '2_4': interpolation(resample_fixed(t2_curves['2']['4']['com_vt'], 100)[0]).values,                    
    '2_5': interpolation(resample_fixed(t2_curves['2']['5']['com_vt'], 100)[0]).values, 
    '3_1': interpolation(resample_fixed(t2_curves['3']['1']['com_vt'], 100)[0]).values, '3_2': interpolation(resample_fixed(t2_curves['3']['2']['com_vt'], 100)[0]).values,  
    '3_3': interpolation(resample_fixed(t2_curves['3']['3']['com_vt'], 100)[0]).values, '3_4': interpolation(resample_fixed(t2_curves['3']['4']['com_vt'], 100)[0]).values,
    '3_5': interpolation(resample_fixed(t2_curves['3']['5']['com_vt'], 100)[0]).values,                   
    '4_1': interpolation(resample_fixed(t2_curves['4']['1']['com_vt'], 100)[0]).values, '4_2': interpolation(resample_fixed(t2_curves['4']['2']['com_vt'], 100)[0]).values, 
    '4_3': interpolation(resample_fixed(t2_curves['4']['3']['com_vt'], 100)[0]).values, '4_4': interpolation(resample_fixed(t2_curves['4']['4']['com_vt'], 100)[0]).values,                    
    '4_5': interpolation(resample_fixed(t2_curves['4']['5']['com_vt'], 100)[0]).values,                    
    '5_1': interpolation(resample_fixed(t2_curves['5']['1']['com_vt'], 100)[0]).values, '5_2': interpolation(resample_fixed(t2_curves['5']['2']['com_vt'], 100)[0]).values,                    
    '5_3': interpolation(resample_fixed(t2_curves['5']['3']['com_vt'], 100)[0]).values, '5_4': interpolation(resample_fixed(t2_curves['5']['4']['com_vt'], 100)[0]).values,                    
    '6_1': interpolation(resample_fixed(t2_curves['6']['1']['com_vt'], 100)[0]).values, '6_2': interpolation(resample_fixed(t2_curves['6']['2']['com_vt'], 100)[0]).values,                    
    '6_3': interpolation(resample_fixed(t2_curves['6']['3']['com_vt'], 100)[0]).values, '6_4': interpolation(resample_fixed(t2_curves['6']['4']['com_vt'], 100)[0]).values,                    
    '6_5': interpolation(resample_fixed(t2_curves['6']['5']['com_vt'], 100)[0]).values,                    
    '7_1': interpolation(resample_fixed(t2_curves['7']['1']['com_vt'], 100)[0]).values, '7_2': interpolation(resample_fixed(t2_curves['7']['2']['com_vt'], 100)[0]).values,                    
    '7_3': interpolation(resample_fixed(t2_curves['7']['3']['com_vt'], 100)[0]).values, '7_4': interpolation(resample_fixed(t2_curves['7']['4']['com_vt'], 100)[0]).values,                    
    '7_5': interpolation(resample_fixed(t2_curves['7']['5']['com_vt'], 100)[0]).values,                   
    '8_1': interpolation(resample_fixed(t2_curves['8']['1']['com_vt'], 100)[0]).values, '8_2': interpolation(resample_fixed(t2_curves['8']['2']['com_vt'], 100)[0]).values,                    
    '8_3': interpolation(resample_fixed(t2_curves['8']['3']['com_vt'], 100)[0]).values, '8_4': interpolation(resample_fixed(t2_curves['8']['4']['com_vt'], 100)[0]).values,                    
    '8_5': interpolation(resample_fixed(t2_curves['8']['5']['com_vt'], 100)[0]).values,                    
    '9_1': interpolation(resample_fixed(t2_curves['9']['1']['com_vt'], 100)[0]).values, '9_2': interpolation(resample_fixed(t2_curves['9']['2']['com_vt'], 100)[0]).values,                   
    '9_3': interpolation(resample_fixed(t2_curves['9']['3']['com_vt'], 100)[0]).values, '9_4': interpolation(resample_fixed(t2_curves['9']['4']['com_vt'], 100)[0]).values,                   
    '9_5': interpolation(resample_fixed(t2_curves['9']['5']['com_vt'], 100)[0]).values,                   
    '10_1': interpolation(resample_fixed(t2_curves['10']['1']['com_vt'], 100)[0]).values, '10_2': interpolation(resample_fixed(t2_curves['10']['2']['com_vt'], 100)[0]).values, 
    '10_3': interpolation(resample_fixed(t2_curves['10']['3']['com_vt'], 100)[0]).values, '10_4': interpolation(resample_fixed(t2_curves['10']['4']['com_vt'], 100)[0]).values, 
    '11_1': interpolation(resample_fixed(t2_curves['11']['1']['com_vt'], 100)[0]).values, '11_2': interpolation(resample_fixed(t2_curves['11']['2']['com_vt'], 100)[0]).values, 
    '11_3': interpolation(resample_fixed(t2_curves['11']['3']['com_vt'], 100)[0]).values, '11_4': interpolation(resample_fixed(t2_curves['11']['4']['com_vt'], 100)[0]).values, 
    '11_5': interpolation(resample_fixed(t2_curves['11']['5']['com_vt'], 100)[0]).values, 
    '12_1': interpolation(resample_fixed(t2_curves['12']['1']['com_vt'], 100)[0]).values, '12_2': interpolation(resample_fixed(t2_curves['12']['2']['com_vt'], 100)[0]).values,
    '12_3': interpolation(resample_fixed(t2_curves['12']['3']['com_vt'], 100)[0]).values
    })

com_vt = res_com_2.mean(numeric_only=True, axis=1)
com_vt_std = res_com_2.std(numeric_only=True, axis=1)
com_vt_diff = com_vt.diff() * 200
com_vt_diff_abs = abs(com_vt_diff)

peaks, val = find_peaks(com_vt_diff_abs, height=com_vt_diff_abs.max()/2) 

plt.figure()
plt.plot(com_vt_diff)
plt.plot(peaks, com_vt_diff[peaks], "x")
plt.plot(np.zeros_like(com_vt_diff), "--", color="gray")

local_minima = argrelextrema(np.array(com_vt_diff_abs), np.less)[0]
plt.plot(local_minima, com_vt_diff[local_minima], 'x', label='peaks')
events = np.array([1, local_minima[0], 99])
plt.plot(events, com_vt_diff[events], 'x', label='peaks')
events_2 = np.array([0, local_minima[0], 100])

curve_registration_params_2 = {'res_com_mean': com_vt,'res_com_std': com_vt_std,'events': events_2}
# with open('/Volumes/Seagate/biomech_analysis/clustering/'+ 'curve_registration_params_2.pkl', 'wb') as handle:
#     pickle.dump(curve_registration_params, handle, protocol=pickle.HIGHEST_PROTOCOL) 


# with open('/Volumes/Seagate/biomech_analysis/clustering/'+ 'curve_registration_params_2.pkl', 'rb') as f:
#     params_2 = pickle.load(f)
params_2 = curve_registration_params_2
com_vt_registered_2, com_ap_registered_2, cop_ap_registered_2, cop_ml_registered_2 = {}, {}, {}, {}
com_vt_vel_registered_2, com_ap_vel_registered_2, cop_ap_vel_registered_2, cop_ml_vel_registered_2 = {}, {}, {}, {}
trunk_registered_2, knee_registered_2 = {}, {}
fx_2, fy_2, fz_2 = {}, {}, {}
rmalx_registered_2, lmalx_registered_2, rtoez_registered_2, ltoez_registered_2 = {}, {}, {}, {}
rheelx_registered_2, rheely_registered_2, rheelz_registered_2 = {}, {}, {}
lheelx_registered_2, lheely_registered_2, lheelz_registered_2 = {}, {}, {}

from scipy.signal import find_peaks, argrelextrema
subjs = ['1', '2', '3', '4', '5', '6', '7', '8', '9','10', '11', '12']
middle = []
for i in subjs:
    for j in t2_curves[i]:
        if j != 'points' and j!='whole_trunk' and j!='weight':

            com_vt_interp = interpolation(t2_curves[i][j]['com_vt']) / t2[i]['d']
            com_ap_interp = interpolation(t2_curves[i][j]['com_ap']) / t2[i]['d']
            cop_ap_interp = interpolation(t2_curves[i][j]['cop_ap']) / t2[i]['d']
            trunk_interp = interpolation(t2_curves[i][j]['trunk'])
            fx = interpolation(t2_curves[i][j]['fx']) / t2_curves[i]['weight']
            fy = interpolation(t2_curves[i][j]['fy']) / t2_curves[i]['weight']
            fz = interpolation(t2_curves[i][j]['fz']) / t2_curves[i]['weight']

            b, a = signal.butter(4, 20, 'lowpass', fs=200)
            fx_filt = signal.filtfilt(b, a, fx)
            fy_filt = signal.filtfilt(b, a, fy)
            fz_filt = signal.filtfilt(b, a, fz)

            com_vt_interp_vel = com_vt_interp.diff() / 200
            com_ap_interp_vel = com_ap_interp.diff() / 200
            cop_ap_interp_vel = cop_ap_interp.diff() / 200
            cop_ml_interp_vel = cop_ml_interp.diff() / 200

            # b, a = signal.butter(3, 20, 'lowpass', fs=200)
            # cop_ap_interp_vel = signal.filtfilt(b, a, cop_ap_interp_vel)
            # cop_ml_interp_vel = signal.filtfilt(b, a, cop_ml_interp_vel)

            rmalx_interp = interpolation(t2_curves[i][j]['rmal'][:,0])
            lmalx_interp = interpolation(t2_curves[i][j]['lmal'][:,0])
            rtoez_interp = interpolation(t2_curves[i][j]['rtoe'][:,2])
            ltoez_interp = interpolation(t2_curves[i][j]['ltoe'][:,2])
            rheelx_interp = interpolation(t2_curves[i][j]['rheel'][:,0])
            rheely_interp = interpolation(t2_curves[i][j]['rheel'][:,1])
            rheelz_interp = interpolation(t2_curves[i][j]['rheel'][:,2])
            lheelx_interp = interpolation(t2_curves[i][j]['lheel'][:,0])
            lheely_interp = interpolation(t2_curves[i][j]['lheel'][:,1])
            lheelz_interp = interpolation(t2_curves[i][j]['lheel'][:,2])

            if i == '9':
                knee_interp = interpolation(t2_curves[i][j]['knee_left'])
                knee_interp = interpolation(t2_curves[i][j]['knee_left'])
            else:
                knee_interp = interpolation(t2_curves[i][j]['knee_right'])
                knee_interp = interpolation(t2_curves[i][j]['knee_right'])

            com_vt_diff = np.diff(com_vt_interp) * 200
            com_vt_diff_abs = abs(com_vt_diff)
            peaks, val = find_peaks(com_vt_diff_abs, height=com_vt_diff_abs.max()/2) 
            local_minima = argrelextrema(np.array(com_vt_diff_abs), np.less)[0]
            events = [1, local_minima[0], len(com_vt_diff)]
            middle.append(local_minima[0])

            # print(events)
            com_vt_vel_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(com_vt_interp_vel),events, params_2['events'])
            com_ap_vel_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(com_ap_interp_vel),events, params_2['events'])
            cop_ap_vel_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(cop_ap_interp_vel),events, params_2['events'])
            cop_ml_vel_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(cop_ml_interp_vel),events, params_2['events'])
            com_vt_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(com_vt_interp),events, params_2['events'])
            com_ap_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(com_ap_interp),events, params_2['events'])
            cop_ap_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(cop_ap_interp),events, params_2['events'])
            cop_ml_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(cop_ml_interp),events, params_2['events'])
            trunk_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(trunk_interp),events, params_2['events'])
            knee_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(knee_interp),events, params_2['events'])
            fx_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(fx_filt),events, params_2['events'])
            fy_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(fy_filt),events, params_2['events'])
            fz_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(fz_filt),events, params_2['events'])
            rmalx_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(rmalx_interp),events, params_2['events'])
            lmalx_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(lmalx_interp),events, params_2['events'])
            rtoez_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(rtoez_interp),events, params_2['events'])
            ltoez_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(ltoez_interp),events, params_2['events'])
            rheelx_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(rheelx_interp),events, params_2['events'])
            rheely_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(rheely_interp),events, params_2['events'])
            rheelz_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(rheelz_interp),events, params_2['events'])
            lheelx_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(lheelx_interp),events, params_2['events'])
            lheely_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(lheely_interp),events, params_2['events'])
            lheelz_registered_2['{}{}'.format(i,j)] = curve_registration(pd.DataFrame(lheelz_interp),events, params_2['events'])

com_vt_registered_2, com_ap_registered_2 = pd.DataFrame(com_vt_registered_2), pd.DataFrame(com_ap_registered_2)
cop_ap_registered_2 = pd.DataFrame(cop_ap_registered_2)
cop_ml_registered_2 = pd.DataFrame(cop_ml_registered_2)
trunk_registered_2, knee_registered_2 = pd.DataFrame(trunk_registered_2), pd.DataFrame(knee_registered_2)
fx_2, fy_2,fz_2  = pd.DataFrame(fx_2), pd.DataFrame(fy_2), pd.DataFrame(fz_2)

com_vt_vel_registered_2, com_ap_vel_registered_2 = pd.DataFrame(com_vt_vel_registered_2), pd.DataFrame(com_ap_vel_registered_2)
cop_ap_vel_registered_2 = pd.DataFrame(cop_ap_vel_registered_2)
cop_ml_vel_registered_2 = pd.DataFrame(cop_ml_vel_registered_2)

com_vt_registered_2 -= com_vt_registered_2.iloc[0]
com_ap_registered_2 -= com_ap_registered_2.iloc[0]
cop_ap_registered_2 -= cop_ap_registered_2.iloc[0]
cop_ml_registered_2 -= cop_ml_registered_2.iloc[0]
trunk_registered_2 -= trunk_registered_2.iloc[0]
knee_registered_2 -= knee_registered_2.iloc[0]

rmalx_registered_2 = pd.DataFrame(rmalx_registered_2)
lmalx_registered_2 = pd.DataFrame(lmalx_registered_2)
rtoez_registered_2 = pd.DataFrame(rtoez_registered_2)
ltoez_registered_2 = pd.DataFrame(ltoez_registered_2)
rheelx_registered_2 = pd.DataFrame(rheelx_registered_2)
rheely_registered_2 = pd.DataFrame(rheely_registered_2)
rheelz_registered_2 = pd.DataFrame(rheelz_registered_2)
lheelx_registered_2 = pd.DataFrame(lheelx_registered_2)
lheely_registered_2 = pd.DataFrame(lheely_registered_2)
lheelz_registered_2 = pd.DataFrame(lheelz_registered_2)

# BOS

In [10]:
def bos(feet, normalized=True):

    mr = feet['rheel_y'][0] *1.2
    ml = feet['lheel_y'][0] *1.2

    # feet['rheel_z'][abs(feet['rheel_y'])>abs(mr)] = 0
    # feet['lheel_z'][abs(feet['lheel_y'])>abs(ml)] = 0
    # feet['rmal_x'][abs(feet['rheel_y'])>abs(mr)] = 0
    # feet['lmal_x'][abs(feet['lheel_y'])>abs(ml)] = 0

    # plt.plot(feet['rheel_z'])
    # plt.plot(feet['lheel_z'])

    base_right_triangle = abs(feet['rtoe_z'] - feet['rheel_z'])
    height_right_triangle = abs(feet['rmal_x'] - feet['rheel_x'])
    right_triangle = (base_right_triangle *height_right_triangle) / 2

    base_left_triangle = abs(feet['ltoe_z'] - feet['lheel_z'])
    height_left_triangle = abs(feet['lmal_x'] - feet['lheel_x'])
    left_triangle = abs((base_left_triangle *height_left_triangle) / 2)

    rectangule_height = abs(feet['rmal_x'] - feet['lmal_x'])
    rectangule_base = abs((base_right_triangle+base_left_triangle)/2)
    rectangule = rectangule_base * rectangule_height

    right_triangle[abs(feet['rheel_y'])>abs(mr)] = 0
    left_triangle[abs(feet['lheel_y'])>abs(ml)] = 0
    rectangule[abs(feet['rheel_y'])>abs(mr)] = abs(feet['rtoe_z'] - feet['ltoe_z']) * .05
    rectangule[abs(feet['lheel_y'])>abs(ml)] = abs(feet['rtoe_z'] - feet['ltoe_z']) * .05


    area = right_triangle + left_triangle + rectangule


    if normalized:
        area /= rectangule_base

    b, a = signal.butter(1, 20, 'lowpass', fs=200)
    area = signal.filtfilt(b, a, area)

    return area

# df_feet = pd.DataFrame({ 'rheel_x':rheelx_registered_1['101'], 'rheel_y':rheely_registered_1['101'], 'rheel_z':rheelz_registered_1['101'], 
#                         'rmal_x':rmalx_registered_1['101'],  'rtoe_z':rtoez_registered_1['101'],
#                         'lheel_x':lheelx_registered_1['101'], 'lheel_y':lheely_registered_1['101'], 'lheel_z':lheelz_registered_1['101'], 
#                         'lmal_x':lmalx_registered_1['101'],  'ltoe_z':ltoez_registered_1['101'],})
# area = bos(df_feet)
# plt.plot(area)

bos_1 = {}
for i in rheelx_registered_1:
    df_feet = pd.DataFrame({ 'rheel_x':rheelx_registered_1[i], 'rheel_y':rheely_registered_1[i], 'rheel_z':rheelz_registered_1[i], 
                        'rmal_x':rmalx_registered_1[i],  'rtoe_z':rtoez_registered_1[i],
                        'lheel_x':lheelx_registered_1[i], 'lheel_y':lheely_registered_1[i], 'lheel_z':lheelz_registered_1[i], 
                        'lmal_x':lmalx_registered_1[i],  'ltoe_z':ltoez_registered_1[i],})
    bos_1['{}'.format(i)] = bos(df_feet)
bos_1 = pd.DataFrame(bos_1)

bos_2 = {}
for i in rheelx_registered_2:
    df_feet = pd.DataFrame({ 'rheel_x':rheelx_registered_2[i], 'rheel_y':rheely_registered_2[i], 'rheel_z':rheelz_registered_2[i], 
                        'rmal_x':rmalx_registered_2[i],  'rtoe_z':rtoez_registered_2[i],
                        'lheel_x':lheelx_registered_2[i], 'lheel_y':lheely_registered_2[i], 'lheel_z':lheelz_registered_2[i], 
                        'lmal_x':lmalx_registered_2[i],  'ltoe_z':ltoez_registered_2[i],})
    bos_2['{}'.format(i)] = bos(df_feet)
bos_2 = pd.DataFrame(bos_2)

# Clustering

## Unimanual

### Stereo

In [11]:
subj_1 =['1','1', '2','2','2','2','2','3','3','3','4','4','4','4','4','5','5','5','5','5','6','6','6','6','6',
                            '7','7','7','7','7','8','8','8','8','8','9','9','9','9','9', '10','10','11','11','11','11','11', '12']

res_1 = {'1_1_t': trunk_registered_1['11'], '1_1_k': knee_registered_1['11'],
                    '1_2_t': trunk_registered_1['12'], '1_2_k': knee_registered_1['12'],
                    '2_1_t': trunk_registered_1['21'], '2_1_k': knee_registered_1['21'],
                    '2_2_t': trunk_registered_1['22'], '2_2_k': knee_registered_1['22'],
                    '2_3_t': trunk_registered_1['23'], '2_3_k': knee_registered_1['23'],
                    '2_4_t': trunk_registered_1['24'], '2_4_k': knee_registered_1['24'],
                    '2_5_t': trunk_registered_1['25'], '2_5_k': knee_registered_1['25'],
                    '3_1_t': trunk_registered_1['31'], '3_1_k': knee_registered_1['31'],
                    '3_2_t': trunk_registered_1['32'], '3_2_k': knee_registered_1['32'],
                    '3_3_t': trunk_registered_1['33'], '3_3_k': knee_registered_1['33'],
                    '4_1_t': trunk_registered_1['41'], '4_1_k': knee_registered_1['41'],
                    '4_2_t': trunk_registered_1['42'], '4_2_k': knee_registered_1['42'],
                    '4_3_t': trunk_registered_1['43'], '4_3_k': knee_registered_1['43'],
                    '4_4_t': trunk_registered_1['44'], '4_4_k': knee_registered_1['44'],
                    '4_5_t': trunk_registered_1['45'], '4_5_k': knee_registered_1['45'],
                    '5_1_t': trunk_registered_1['51'], '5_1_k': knee_registered_1['51'],
                    '5_2_t': trunk_registered_1['52'], '5_2_k': knee_registered_1['52'],
                    '5_3_t': trunk_registered_1['53'], '5_3_k': knee_registered_1['53'],
                    '5_4_t': trunk_registered_1['54'], '5_4_k': knee_registered_1['54'],
                    '5_5_t': trunk_registered_1['55'], '5_5_k': knee_registered_1['55'],
                    '6_1_t': trunk_registered_1['61'], '6_1_k': knee_registered_1['61'],
                    '6_2_t': trunk_registered_1['62'], '6_2_k': knee_registered_1['62'],
                    '6_3_t': trunk_registered_1['63'], '6_3_k': knee_registered_1['63'],
                    '6_4_t': trunk_registered_1['64'], '6_4_k': knee_registered_1['64'],
                    '6_5_t': trunk_registered_1['65'], '6_5_k': knee_registered_1['65'],
                    '7_1_t': trunk_registered_1['71'], '7_1_k': knee_registered_1['71'],
                    '7_2_t': trunk_registered_1['72'], '7_2_k': knee_registered_1['72'],
                    '7_3_t': trunk_registered_1['73'], '7_3_k': knee_registered_1['73'],
                    '7_4_t': trunk_registered_1['74'], '7_4_k': knee_registered_1['74'],
                    '7_5_t': trunk_registered_1['75'], '7_5_k': knee_registered_1['75'],
                    '8_1_t': trunk_registered_1['81'], '8_1_k': knee_registered_1['81'],
                    '8_2_t': trunk_registered_1['82'], '8_2_k': knee_registered_1['82'],
                    '8_3_t': trunk_registered_1['83'], '8_3_k': knee_registered_1['83'],
                    '8_4_t': trunk_registered_1['84'], '8_4_k': knee_registered_1['84'],
                    '8_5_t': trunk_registered_1['85'], '8_5_k': knee_registered_1['85'],
                    '9_1_t': trunk_registered_1['91'], '9_1_k': knee_registered_1['91'],
                    '9_2_t': trunk_registered_1['92'], '9_2_k': knee_registered_1['92'],
                    '9_3_t': trunk_registered_1['93'], '9_3_k': knee_registered_1['93'],
                    '9_4_t': trunk_registered_1['94'], '9_4_k': knee_registered_1['94'],
                    '9_5_t': trunk_registered_1['95'], '9_5_k': knee_registered_1['95'],
                    '10_1_t': trunk_registered_1['101'], '10_1_k': knee_registered_1['101'],
                    '10_2_t': trunk_registered_1['102'], '10_2_k': knee_registered_1['102'],
                    '11_1_t': trunk_registered_1['111'], '11_1_k': knee_registered_1['111'],
                    '11_2_t': trunk_registered_1['112'], '11_2_k': knee_registered_1['111'],
                    '11_3_t': trunk_registered_1['113'], '11_3_k': knee_registered_1['113'],
                    '11_4_t': trunk_registered_1['114'], '11_4_k': knee_registered_1['114'],
                    '11_5_t': trunk_registered_1['115'], '11_5_k': knee_registered_1['115'],
                    '12_1_t': trunk_registered_1['121'], '12_1_k': knee_registered_1['121']}

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from tslearn.clustering import TimeSeriesKMeans
from tslearn.utils import to_time_series_dataset

t11, t12 = np.vstack([res_1['1_1_t'],res_1['1_1_k']]).T , np.vstack([res_1['1_2_t'],res_1['1_2_k']]).T  
t21, t22, t23 = np.vstack([res_1['2_1_t'],res_1['2_1_k']]).T , np.vstack([res_1['2_2_t'],res_1['2_2_k']]).T , np.vstack([res_1['2_3_t'],res_1['2_3_k']]).T  
t24, t25 = np.vstack([res_1['2_4_t'],res_1['2_4_k']]).T , np.vstack([res_1['2_5_t'],res_1['2_5_k']]).T  
t31, t32, t33 = np.vstack([res_1['3_1_t'],res_1['3_1_k']]).T , np.vstack([res_1['3_2_t'],res_1['3_2_k']]).T , np.vstack([res_1['3_3_t'],res_1['3_3_k']]).T  
t41, t42, t43 = np.vstack([res_1['4_1_t'],res_1['4_1_k']]).T , np.vstack([res_1['4_2_t'],res_1['4_2_k']]).T , np.vstack([res_1['4_3_t'],res_1['4_3_k']]).T  
t44, t45 = np.vstack([res_1['4_4_t'],res_1['4_4_k']]).T , np.vstack([res_1['4_5_t'],res_1['4_5_k']]).T  
t51, t52, t53 = np.vstack([res_1['5_1_t'],res_1['5_1_k']]).T , np.vstack([res_1['5_2_t'],res_1['5_2_k']]).T ,  np.vstack([res_1['5_3_t'],res_1['5_3_k']]).T  
t54, t55 = np.vstack([res_1['5_4_t'],res_1['5_4_k']]).T , np.vstack([res_1['5_5_t'],res_1['5_5_k']]).T  
t61, t62, t63 = np.vstack([res_1['6_1_t'],res_1['6_1_k']]).T , np.vstack([res_1['6_2_t'],res_1['6_2_k']]).T ,  np.vstack([res_1['6_3_t'],res_1['6_3_k']]).T  
t64, t65 = np.vstack([res_1['6_4_t'],res_1['6_4_k']]).T , np.vstack([res_1['6_5_t'],res_1['6_5_k']]).T  
t71, t72, t73 = np.vstack([res_1['7_1_t'],res_1['7_1_k']]).T , np.vstack([res_1['7_2_t'],res_1['7_2_k']]).T ,  np.vstack([res_1['7_3_t'],res_1['7_3_k']]).T  
t74, t75 = np.vstack([res_1['7_4_t'],res_1['7_4_k']]).T , np.vstack([res_1['7_5_t'],res_1['7_5_k']]).T  
t81, t82, t83 = np.vstack([res_1['8_1_t'],res_1['8_1_k']]).T , np.vstack([res_1['8_2_t'],res_1['8_2_k']]).T ,  np.vstack([res_1['8_3_t'],res_1['8_3_k']]).T  
t84, t85 = np.vstack([res_1['8_4_t'],res_1['8_4_k']]).T , np.vstack([res_1['8_5_t'],res_1['8_5_k']]).T  
t91, t92, t93 = np.vstack([res_1['9_1_t'],res_1['9_1_k']]).T , np.vstack([res_1['9_2_t'],res_1['9_2_k']]).T ,  np.vstack([res_1['9_3_t'],res_1['9_3_k']]).T  
t94, t95 = np.vstack([res_1['9_4_t'],res_1['9_4_k']]).T , np.vstack([res_1['9_5_t'],res_1['9_5_k']]).T  
t101, t102 = np.vstack([res_1['10_1_t'],res_1['10_1_k']]).T , np.vstack([res_1['10_2_t'],res_1['10_2_k']]).T 
t111, t112, t113 = np.vstack([res_1['11_1_t'],res_1['11_1_k']]).T , np.vstack([res_1['11_2_t'],res_1['11_2_k']]).T ,  np.vstack([res_1['11_3_t'],res_1['11_3_k']]).T  
t114, t115 = np.vstack([res_1['11_4_t'],res_1['11_4_k']]).T , np.vstack([res_1['11_5_t'],res_1['11_5_k']]).T  
t121 = np.vstack([res_1['12_1_t'],res_1['12_1_k']]).T   

X = np.array([t11, t12, t21, t22, t23, t24, t25, t31, t32, t33, t41, t42, t43, t44, t45, t51, t52, t53, t54, t55, t61, t62, t63, t64, t65,
              t71, t72, t73, t74, t75, t81, t82, t83, t84, t85, t91, t92, t93, t94, t95, t101, t102, t111, t112, t113, t114, t115, t121]) 

n_features = 2
n_samples = len(X)

from kneed import KneeLocator  # Import the KneeLocator from kneed

# Range of clusters to try
cluster_range = range(1, 11)  # Try from 1 to 10 clusters
inertia_values = []
# Run TimeSeriesKMeans for different cluster sizes
for n_clusters in cluster_range:
    model_uni_stereo = TimeSeriesKMeans(n_clusters=n_clusters, metric="euclidean", verbose=False)
    model_uni_stereo.fit(X)
    inertia_values.append(model_uni_stereo.inertia_)

# Use KneeLocator to find the optimal "knee" (elbow point) in the plot
kneedle = KneeLocator(cluster_range, inertia_values, curve='convex', direction='decreasing')

# Get the "knee" point, i.e., the optimal number of clusters
n_clusters = kneedle.elbow
print(f"Optimal number of clusters: {n_clusters}")

n_clusters = 2

seed = 0
np.random.seed(seed)
model_uni_stereo = TimeSeriesKMeans(n_clusters=n_clusters, n_init=5, random_state=seed).fit(X)
y_pred = model_uni_stereo.fit_predict(X)
centers = model_uni_stereo.cluster_centers_

plt.style.use("default")
# Step 3: Plot the cluster centers (these represent the "average" time-series for each cluster)
f, axs = plt.subplots(1,3,figsize=(20, 5))
f.suptitle('Unimanual lifting',fontsize=20)
# colors = ['b', 'm','g']
# names = ['Stoop','Mixed', 'Squat']
# clusters_idx = [2, 1,0]

colors = ['m','g']
names = ['Hybrid', 'Squat']
clusters_idx = [1,0]

# Plot each cluster center (the mean time-series in each cluster)
for cluster_idx, lab, c in zip(clusters_idx, names, colors):
    axs[0].plot(-model_uni_stereo.cluster_centers_[cluster_idx][:, 0], label=f" {lab} - Trunk", color=c, linestyle='solid')  # Feature 1
    axs[0].plot(model_uni_stereo.cluster_centers_[cluster_idx][:, 1], label=f" {lab} - Knee", color=c, linestyle='dashed')  # Feature 2

# axs[0].set_title("Cluster Centers with TimeSeriesKMeans (Multivariate Time Series)")
axs[0].set_title("Cluster Centers", size=14)
axs[0].set_xlabel("Movement Cycle (%)", size=14)
axs[0].set_ylabel("Flexion angles (deg)", size=14)
axs[0].legend(fontsize=14)

# Step 4: Print predicted cluster labels for the first few time-series
print("Predicted cluster labels for all time-series:", y_pred)
# print(pd.DataFrame({'subj':subj, 'cluster':y_pred}))

LABEL_COLOR_MAP = {0 : 'g',
                   1 : 'm',
                   2 : 'b',
                   }

label_color = [LABEL_COLOR_MAP[l] for l in y_pred]

# Optionally, visualize how the time-series are grouped into clusters
labeled_colors = set()

for i, c in zip(range(n_samples), label_color):
    if c not in labeled_colors:
        if c == 'm': lb = 'Hybrid'
        else: lb = 'Squat'

        axs[1].plot(-X[i, :, 0], color=label_color[i], label=lb)
        axs[2].plot(X[i, :, 1], color=label_color[i], label=lb)
        labeled_colors.add(c)
    else:
        axs[1].plot(-X[i, :, 0], color=label_color[i])
        axs[2].plot(X[i, :, 1], color=label_color[i])
axs[1].set_title("Trunk flexion angle grouped by Strategies", size=14)
axs[1].set_xlabel("Movement Cycle (%)", size=14)

axs[2].set_title("Knee flexion angles grouped by Strategies")
axs[2].set_xlabel("Movement Cycle (%)", size=14)

axs[0].set_ylim([-30,150])
axs[1].set_ylim([-30,150])
axs[2].set_ylim([-30,150])

axs[1].legend(fontsize=14)
axs[2].legend(fontsize=14)

axs[1].set_ylabel("Trunk flexion angles (deg)", size=14)
axs[2].set_ylabel("Knee flexion angles (deg)", size=14)

plt.tight_layout(rect=[0, 0, 1, 0.99])  # reserve space for suptitle

y_pred_1 = y_pred
pd.DataFrame({'subj_1':subj_1, 'results':y_pred_1})

/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/pyth

Optimal number of clusters: 3


/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Predicted cluster labels for all time-series: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 1 1 1 1 0 1 0 0 0 0 0 1 0 0 0 0 0
 0 0 0 0 0 1 1 1 1 1 1]


,subj_1,results
0,1,1
1,1,1
2,2,1
3,2,1
4,2,1
5,2,1
6,2,1
7,3,1
8,3,1
9,3,1


In [13]:
# print elbow
plt.figure(figsize=(8,5))
plt.plot(cluster_range, inertia_values, 'o-', color='blue')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method for the Number of Clusters')
plt.axvline(x=n_clusters, color='red', linestyle='--', label=f'Optimal clusters = {n_clusters}')
plt.axvline(x=2, color='black', linestyle='-.', label=f'Considered clusters = 2')
plt.legend()
plt.show()

### IMU

In [14]:
# 2, 3
init = [1050, 1597]
ending = [1309, 1840]
reps = ['1', '2']
angles_1 = {'11':[], '12':[]}
max_angles_1 = []

for j, i, e in zip(reps, init, ending):
    sf = sensor_fusion(t1_curves['1']['whole_trunk']['acc'][['ax','ay','az']][i:e], t1_curves['1']['whole_trunk']['gyro'][['gx','gy','gz']][i:e])
    angles_1['1{}'.format(j)]= resample_fixed(pd.DataFrame(-sf['eu'][:,1]), 100).values
    max_angles_1.append(max(abs(angles_1['1{}'.format(j)])))
    
subjs_1_imu = ['2', '3', '4', '5', '6', '7', '8', '9','10', '11', '12']

for i in subjs_1_imu:
    for j in t1_curves[i]:
        if j!= 'points' and j!= 'whole_trunk' and j!='weight':
            if "trunk" in t1_curves[i][j]:
                sf = sensor_fusion(t1_curves[i][j]['imu']['acc'].values, t1_curves[i][j]['imu']['gyro'].values)
                angles_1['{}{}'.format(i,j)]= resample_fixed(pd.DataFrame(-sf['eu'][:,1]), 100).values
                max_angles_1.append(max(abs(angles_1['{}{}'.format(i,j)])))
                # plt.plot(sf['eu'][:,1])
                # print('{} - {}'.format(i, max(abs(angles_1['{}{}'.format(i,j)]))))


In [81]:
X = np.array([
    angles_1['11'], angles_1['12'], 
              angles_1['21'], angles_1['23'], angles_1['23'], angles_1['24'], angles_1['25'], 
              angles_1['31'], angles_1['32'], angles_1['33'], 
              angles_1['41'], angles_1['42'], angles_1['43'], angles_1['44'], angles_1['45'], 
              angles_1['51'], angles_1['52'], angles_1['53'], angles_1['54'], angles_1['55'], 
              angles_1['61'], angles_1['62'], angles_1['63'], angles_1['64'], angles_1['65'],
              angles_1['71'], angles_1['72'], angles_1['73'], angles_1['74'], angles_1['75'], 
              angles_1['81'], angles_1['82'], angles_1['83'], angles_1['84'], angles_1['85'], 
              angles_1['91'], angles_1['92'], angles_1['93'], angles_1['94'], angles_1['95'], 
              angles_1['101'], angles_1['102'], 
              angles_1['111'], angles_1['112'], angles_1['113'], angles_1['114'], angles_1['115'], 
              angles_1['121']]) 

n_features = 1
# n_clusters = 3       
n_samples = len(X)

from kneed import KneeLocator  # Import the KneeLocator from kneed
seed = 0
np.random.seed(seed)
# Range of clusters to try
cluster_range = range(1, 11)  # Try from 1 to 10 clusters
inertia_values = []
# Run TimeSeriesKMeans for different cluster sizes
for n_clusters in cluster_range:
    model_uni_imu = TimeSeriesKMeans(n_clusters=n_clusters,  n_init=5, random_state=seed, metric="euclidean", verbose=False)
    model_uni_imu.fit(X)
    inertia_values.append(model_uni_imu.inertia_)

# Use KneeLocator to find the optimal "knee" (elbow point) in the plot
kneedle = KneeLocator(cluster_range, inertia_values, curve='convex', direction='decreasing')

# Get the "knee" point, i.e., the optimal number of clusters
n_clusters = kneedle.elbow
print(f"Optimal number of clusters: {n_clusters}")

from tslearn.preprocessing import TimeSeriesScalerMeanVariance, TimeSeriesResampler

n_clusters = 2
X_train = TimeSeriesScalerMeanVariance().fit_transform(X)
model_uni_imu = TimeSeriesKMeans(n_clusters=n_clusters, n_init=5, random_state=seed).fit(X_train)
y_pred_1_imu = model_uni_imu.fit_predict(X)

plt.style.use('default')
# Step 3: Plot the cluster centers (these represent the "average" time-series for each cluster)
f, axs = plt.subplots(1,2,figsize=(20, 6))
f.suptitle('Unimanual lifting', y=0.95, fontsize=20)

# colors = ['b', 'm', 'g']
# clusters_id = [2, 1, 0]
# names = ['Stoop','Hybrid', 'Squat']
colors = ['m', 'g']
clusters_id = [1, 0]
names = ['Hybrid', 'Squat']
# Plot each cluster center (the mean time-series in each cluster)
for cluster_idx, lab, c in zip(clusters_id, names, colors):
    axs[0].plot(-model_uni_imu.cluster_centers_[cluster_idx][:, 0], label=f" {lab} - L5", color=c, linestyle='solid')  # Feature 1

# axs[0].set_title("Cluster Centers with TimeSeriesKMeans (Multivariate Time Series)")
axs[0].set_title("Cluster Centers", size=14)
axs[0].set_xlabel("Movement cycle (%)", size=14)
axs[0].set_ylabel("Flexion angles (deg)", size=14)
axs[0].legend(fontsize=14)

# Step 4: Print predicted cluster labels for the first few time-series
print("Predicted cluster labels for all time-series:", y_pred_1_imu)
# print(pd.DataFrame({'subj':subj, 'cluster':y_pred_1_imu}))

LABEL_COLOR_MAP = { 0 : 'g',
                    1 : 'm',
                   2 : 'b',
                   }

label_color = [LABEL_COLOR_MAP[l] for l in y_pred_1_imu]


# Optionally, visualize how the time-series are grouped into clusters
# labeled_colors = set()
# clusters = [1,0]
# for i, c in zip(range(n_samples), label_color):
#     if c not in labeled_colors:
#         if c == 'm': lb = 'Hybrid'
#         else: lb = 'Squat'

#         axs[1].plot(-X[i, :, 0], color=label_color[i], label=lb)
#         labeled_colors.add(c)
#     else:
#         axs[1].plot(-X[i, :, 0], color=label_color[i])

# Ensure legend order: first Hybrid (m), then Squat (g)
clusters = [1, 0]        # cluster ids
labels = ['Hybrid', 'Squat']
colors = ['m', 'g']

# Plot each cluster's first example with label
plotted = set()
for cluster_id, label, color in zip(clusters, labels, colors):
    # find first index in this cluster
    idx = np.where(y_pred_1_imu == cluster_id)[0][0]
    axs[1].plot(-X[idx, :, 0], color=color, label=label)
    plotted.add(idx)

# Plot the remaining time-series without labels
for i in range(n_samples):
    if i not in plotted:
        axs[1].plot(-X[i, :, 0], color=label_color[i])


axs[1].set_title("L5 flexion angles grouped by strategy", size=14)
# axs[1].set_ylabel("Degrees")
axs[0].set_ylim([-10, 80])
axs[1].set_ylim([-10, 80])
axs[1].legend(fontsize=14)

axs[1].set_xlabel("Movement cycle (%)", size=14)
axs[1].set_ylabel("L5 flexion angles (deg)", size=14)
plt.tight_layout(rect=[0, 0, 1, 0.99])  # reserve space for suptitle

# pd.DataFrame({'subj_1':subj_1[2:], 'results':y_pred_1_imu})
pd.DataFrame({'subj_1':subj_1, 'results':y_pred_1_imu})

/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/pyth

Optimal number of clusters: 3


/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Predicted cluster labels for all time-series: [0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 1 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 1]


,subj_1,results
0,1,0
1,1,0
2,2,1
3,2,1
4,2,1
5,2,1
6,2,1
7,3,1
8,3,1
9,3,1


2025-12-02 17:09:17.608 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:09:17.712 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:09:29.308 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:09:29.406 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:09:40.499 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:09:40.595 python[3160:47829] +[CATransaction synchronize] called within transaction


## Bimanual

### Stereo

In [15]:
subj_2 =['1','1', '1','2','2','2','2','2','3','3','3','3','3','4','4','4','4','4','5','5','5','5','6','6','6','6','6',
                            '7','7','7','7','7','8','8','8','8','8','9','9','9','9','9', '10','10','10','10','11','11','11','11','11', '12','12','12']

res_2 = {'1_1_t': trunk_registered_2['11'], '1_1_k': knee_registered_2['11'],'1_2_t': trunk_registered_2['12'], '1_2_k': knee_registered_2['12'],
        '1_3_t': trunk_registered_2['13'], '1_3_k': knee_registered_2['13'],
        '2_1_t': trunk_registered_2['21'], '2_1_k': knee_registered_2['21'],'2_2_t': trunk_registered_2['22'], '2_2_k': knee_registered_2['22'],
        '2_3_t': trunk_registered_2['23'], '2_3_k': knee_registered_2['23'],'2_4_t': trunk_registered_2['24'], '2_4_k': knee_registered_2['24'],
        '2_5_t': trunk_registered_2['25'], '2_5_k': knee_registered_2['25'],
        '3_1_t': trunk_registered_2['31'], '3_1_k': knee_registered_2['31'],'3_2_t': trunk_registered_2['32'], '3_2_k': knee_registered_2['32'],
        '3_3_t': trunk_registered_2['33'], '3_3_k': knee_registered_2['33'],'3_4_t': trunk_registered_2['34'], '3_4_k': knee_registered_2['34'],
        '3_5_t': trunk_registered_2['35'], '3_5_k': knee_registered_2['35'],
        '4_1_t': trunk_registered_2['41'], '4_1_k': knee_registered_2['41'], '4_2_t': trunk_registered_2['42'], '4_2_k': knee_registered_2['42'],
        '4_3_t': trunk_registered_2['43'], '4_3_k': knee_registered_2['43'],'4_4_t': trunk_registered_2['44'], '4_4_k': knee_registered_2['44'],
        '4_5_t': trunk_registered_2['45'], '4_5_k': knee_registered_2['45'],
        '5_1_t': trunk_registered_2['51'], '5_1_k': knee_registered_2['51'], '5_2_t': trunk_registered_2['52'], '5_2_k': knee_registered_2['52'],
        '5_3_t': trunk_registered_2['53'], '5_3_k': knee_registered_2['53'], '5_4_t': trunk_registered_2['54'], '5_4_k': knee_registered_2['54'],
        '6_1_t': trunk_registered_2['61'], '6_1_k': knee_registered_2['61'],'6_2_t': trunk_registered_2['62'], '6_2_k': knee_registered_2['62'],
        '6_3_t': trunk_registered_2['63'], '6_3_k': knee_registered_2['63'],'6_4_t': trunk_registered_2['64'], '6_4_k': knee_registered_2['64'],
        '6_5_t': trunk_registered_2['65'], '6_5_k': knee_registered_2['65'],
        '7_1_t': trunk_registered_2['71'], '7_1_k': knee_registered_2['71'],'7_2_t': trunk_registered_2['72'], '7_2_k': knee_registered_2['72'],
        '7_3_t': trunk_registered_2['73'], '7_3_k': knee_registered_2['73'],'7_4_t': trunk_registered_2['74'], '7_4_k': knee_registered_2['74'],
        '7_5_t': trunk_registered_2['75'], '7_5_k': knee_registered_2['75'],
        '8_1_t': trunk_registered_2['81'], '8_1_k': knee_registered_2['81'],'8_2_t': trunk_registered_2['82'], '8_2_k': knee_registered_2['82'],
        '8_3_t': trunk_registered_2['83'], '8_3_k': knee_registered_2['83'],'8_4_t': trunk_registered_2['84'], '8_4_k': knee_registered_2['84'],
        '8_5_t': trunk_registered_2['85'], '8_5_k': knee_registered_2['85'],
        '9_1_t': trunk_registered_2['91'], '9_1_k': knee_registered_2['91'],'9_2_t': trunk_registered_2['92'], '9_2_k': knee_registered_2['92'],
        '9_3_t': trunk_registered_2['93'], '9_3_k': knee_registered_2['93'],'9_4_t': trunk_registered_2['94'], '9_4_k': knee_registered_2['94'],
        '9_5_t': trunk_registered_2['95'], '9_5_k': knee_registered_2['95'],
        '10_1_t': trunk_registered_2['101'], '10_1_k': knee_registered_2['101'], '10_2_t': trunk_registered_2['102'], '10_2_k': knee_registered_2['102'],
        '10_3_t': trunk_registered_2['103'], '10_3_k': knee_registered_2['103'],'10_4_t': trunk_registered_2['104'], '10_4_k': knee_registered_2['104'],
        '11_1_t': trunk_registered_2['111'], '11_1_k': knee_registered_2['111'],'11_2_t': trunk_registered_2['112'], '11_2_k': knee_registered_2['111'],
        '11_3_t': trunk_registered_2['113'], '11_3_k': knee_registered_2['113'],'11_4_t': trunk_registered_2['114'], '11_4_k': knee_registered_2['114'],
        '11_5_t': trunk_registered_2['115'], '11_5_k': knee_registered_2['115'],
        '12_1_t': trunk_registered_2['121'], '12_1_k': knee_registered_2['121'],'12_2_t': trunk_registered_2['122'], '12_2_k': knee_registered_2['122'],
        '12_3_t': trunk_registered_2['123'], '12_3_k': knee_registered_2['123']}

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from tslearn.clustering import TimeSeriesKMeans
from tslearn.utils import to_time_series_dataset

t11, t12, t13 = np.vstack([res_2['1_1_t'],res_2['1_1_k']]).T , np.vstack([res_2['1_2_t'],res_2['1_2_k']]).T, np.vstack([res_2['1_3_t'],res_2['1_3_k']]).T 
t21, t22, t23 = np.vstack([res_2['2_1_t'],res_2['2_1_k']]).T , np.vstack([res_2['2_2_t'],res_2['2_2_k']]).T , np.vstack([res_2['2_3_t'],res_2['2_3_k']]).T  
t24, t25 = np.vstack([res_2['2_4_t'],res_2['2_4_k']]).T , np.vstack([res_2['2_5_t'],res_2['2_5_k']]).T  
t31, t32, t33 = np.vstack([res_2['3_1_t'],res_2['3_1_k']]).T , np.vstack([res_2['3_2_t'],res_2['3_2_k']]).T , np.vstack([res_2['3_3_t'],res_2['3_3_k']]).T  
t34, t35 = np.vstack([res_2['3_4_t'],res_2['3_4_k']]).T , np.vstack([res_2['3_5_t'],res_2['3_5_k']]).T 
t41, t42, t43 = np.vstack([res_2['4_1_t'],res_2['4_1_k']]).T , np.vstack([res_2['4_2_t'],res_2['4_2_k']]).T , np.vstack([res_2['4_3_t'],res_2['4_3_k']]).T  
t44, t45 = np.vstack([res_2['4_4_t'],res_2['4_4_k']]).T , np.vstack([res_2['4_5_t'],res_2['4_5_k']]).T  
t51, t52, t53, t54 = np.vstack([res_2['5_1_t'],res_2['5_1_k']]).T , np.vstack([res_2['5_2_t'],res_2['5_2_k']]).T ,  np.vstack([res_2['5_3_t'],res_2['5_3_k']]).T , np.vstack([res_2['5_4_t'],res_2['5_4_k']]).T
t61, t62, t63 = np.vstack([res_2['6_1_t'],res_2['6_1_k']]).T , np.vstack([res_2['6_2_t'],res_2['6_2_k']]).T ,  np.vstack([res_2['6_3_t'],res_2['6_3_k']]).T  
t64, t65 = np.vstack([res_2['6_4_t'],res_2['6_4_k']]).T , np.vstack([res_2['6_5_t'],res_2['6_5_k']]).T  
t71, t72, t73 = np.vstack([res_2['7_1_t'],res_2['7_1_k']]).T , np.vstack([res_2['7_2_t'],res_2['7_2_k']]).T ,  np.vstack([res_2['7_3_t'],res_2['7_3_k']]).T  
t74, t75 = np.vstack([res_2['7_4_t'],res_2['7_4_k']]).T , np.vstack([res_2['7_5_t'],res_2['7_5_k']]).T  
t81, t82, t83 = np.vstack([res_2['8_1_t'],res_2['8_1_k']]).T , np.vstack([res_2['8_2_t'],res_2['8_2_k']]).T ,  np.vstack([res_2['8_3_t'],res_2['8_3_k']]).T  
t84, t85 = np.vstack([res_2['8_4_t'],res_2['8_4_k']]).T , np.vstack([res_2['8_5_t'],res_2['8_5_k']]).T  
t91, t92, t93 = np.vstack([res_2['9_1_t'],res_2['9_1_k']]).T , np.vstack([res_2['9_2_t'],res_2['9_2_k']]).T ,  np.vstack([res_2['9_3_t'],res_2['9_3_k']]).T  
t94, t95 = np.vstack([res_2['9_4_t'],res_2['9_4_k']]).T , np.vstack([res_2['9_5_t'],res_2['9_5_k']]).T  
t101, t102 = np.vstack([res_2['10_1_t'],res_2['10_1_k']]).T , np.vstack([res_2['10_2_t'],res_2['10_2_k']]).T 
t103, t104 = np.vstack([res_2['10_2_t'],res_2['10_2_k']]).T , np.vstack([res_2['10_4_t'],res_2['10_4_k']]).T 
t111, t112, t113 = np.vstack([res_2['11_1_t'],res_2['11_1_k']]).T , np.vstack([res_2['11_2_t'],res_2['11_2_k']]).T ,  np.vstack([res_2['11_3_t'],res_2['11_3_k']]).T  
t114, t115 = np.vstack([res_2['11_4_t'],res_2['11_4_k']]).T , np.vstack([res_2['11_5_t'],res_2['11_5_k']]).T  
t121, t122, t123 = np.vstack([res_2['12_1_t'],res_2['12_1_k']]).T   , np.vstack([res_2['12_2_t'],res_2['12_2_k']]).T   , np.vstack([res_2['12_3_t'],res_2['12_3_k']]).T   

X = np.array([t11, t12,  t13,t21, t22, t23, t24, t25, t31, t32, t33, t34, t35, t41, t42, t43, t44, t45, t51, t52, t53, t54, t61, t62, t63, t64, t65,
              t71, t72, t73, t74, t75, t81, t82, t83, t84, t85, t91, t92, t93, t94, t95, t101, t102, t103, t104, t111, t112, t113, t114, t115, 
              t121, t122, t123]) 

n_features = 2
# n_clusters = 3       
n_samples = len(X)
from kneed import KneeLocator  # Import the KneeLocator from kneed

# Range of clusters to try
cluster_range = range(1, 11)  # Try from 1 to 10 clusters
inertia_values = []
# Run TimeSeriesKMeans for different cluster sizes
for n_clusters in cluster_range:
    model_bi_stereo = TimeSeriesKMeans(n_clusters=n_clusters, metric="euclidean", verbose=False)
    model_bi_stereo.fit(X)
    inertia_values.append(model_bi_stereo.inertia_)

# Use KneeLocator to find the optimal "knee" (elbow point) in the plot
kneedle = KneeLocator(cluster_range, inertia_values, curve='convex', direction='decreasing')

# Get the "knee" point, i.e., the optimal number of clusters
n_clusters = kneedle.elbow
print(f"Optimal number of clusters: {n_clusters}")

seed = 0
np.random.seed(seed)
n_clusters = 2
model_bi_stereo = TimeSeriesKMeans(n_clusters=n_clusters, n_init = 5, random_state=seed).fit(X)
y_pred = model_bi_stereo.fit_predict(X)
plt.style.use("default")

# Step 3: Plot the cluster centers (these represent the "average" time-series for each cluster)
f, axs = plt.subplots(1,3,figsize=(20, 5))
f.suptitle("Bimanual lifting", fontsize=20)
colors = ['m','g']
# Plot each cluster center (the mean time-series in each cluster)
for cluster_idx, lab, c in zip([1,0], ['Hybrid', 'Squat'], colors):
    axs[0].plot(-model_bi_stereo.cluster_centers_[cluster_idx][:, 0], label=f" {lab} - Trunk", color=c, linestyle='solid')  # Feature 1
    axs[0].plot(model_bi_stereo.cluster_centers_[cluster_idx][:, 1], label=f" {lab} - Knee", color=c, linestyle='dashed')  # Feature 2

# axs[0].set_title("Cluster Centers with TimeSeriesKMeans (Multivariate Time Series)")
axs[0].set_title("Cluster Centers", fontsize=14)
axs[0].set_xlabel("Movement cycle (%)")
axs[0].set_ylabel("Flexion angles (deg)")
axs[0].legend(fontsize=14)

# Step 4: Print predicted cluster labels for the first few time-series
print("Predicted cluster labels for all time-series:", y_pred)
# print(pd.DataFrame({'subj':subj, 'cluster':y_pred}))

LABEL_COLOR_MAP = {0 : 'g',
                   1 : 'm'}

label_color = [LABEL_COLOR_MAP[l] for l in y_pred]

# Optionally, visualize how the time-series are grouped into clusters
# for i in range(n_samples):
    # axs[1].plot(X[i, :, 0], color=f"C{y_pred[i]}")  # Color each time-series by its predicted cluster label
    # axs[2].plot(X[i, :, 1], color=f"C{y_pred[i]}")  # Color each time-series by its predicted cluster label
labeled_colors = set()
for i, c in zip(range(n_samples), label_color):
    if c not in labeled_colors:
        if c == 'm': lb = 'Hybrid'
        else: lb = 'Squat'

        axs[1].plot(-X[i, :, 0], color=label_color[i], label=lb)
        axs[2].plot(X[i, :, 1], color=label_color[i], label=lb)
        labeled_colors.add(c)
    else:
        axs[1].plot(-X[i, :, 0], color=label_color[i])
        axs[2].plot(X[i, :, 1], color=label_color[i])

axs[1].set_title("Trunk flexion angle grouped by Strategies")
axs[1].set_xlabel("Movement cycle (%)")

axs[2].set_title("Knee flexion angles grouped by Strategies")
axs[2].set_xlabel("Movement cycle (%)")

axs[0].set_ylim([-20, 150])
axs[1].set_ylim([-20, 150])
axs[2].set_ylim([-20, 150])
axs[1].set_ylabel("Trunk flexion angles (deg)")
axs[2].set_ylabel("Knee flexion angles (deg)")
axs[1].legend(fontsize=14)
axs[2].legend(fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.99])  # reserve space for suptitle

y_pred_2 = y_pred

pd.DataFrame({'subj_2':subj_2, 'results':y_pred})


/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/pyth

Optimal number of clusters: 3


/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Predicted cluster labels for all time-series: [1 1 1 1 1 1 1 1 0 0 0 0 0 1 1 1 1 1 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1]


,subj_2,results
0,1,1
1,1,1
2,1,1
3,2,1
4,2,1
5,2,1
6,2,1
7,2,1
8,3,0
9,3,0


### IMU

In [17]:
# 3,4, 5
init = [1710, 2372, 3030]
ending = [1990, 2639, 3277]
reps = ['1', '2', '3']
angles_2 = {'11':[],'12':[],'13':[]}
max_angles_2 = []

for j, i, e in zip(reps, init, ending):
    sf = sensor_fusion(t2_curves['1']['whole_trunk']['acc'][['ax','ay','az']][i:e], t2_curves['1']['whole_trunk']['gyro'][['gx','gy','gz']][i:e])
    angles_2['1{}'.format(j)]= resample_fixed(pd.DataFrame(-sf['eu'][:,1]), 100).values
    max_angles_2.append(max(abs(angles_2['1{}'.format(j)])))

subjs_2_imu = ['2', '3', '4', '5', '6', '7', '8', '9','10', '11', '12']

# ang_trunk = []
# angles_2 = {}
for i in subjs_2_imu:
    for j in t2_curves[i]:
        if j!='points' and j!='whole_trunk' and j!='weight':
            if "trunk" in t2_curves[i][j]:
                sf = sensor_fusion(t2_curves[i][j]['imu']['acc'].values, t2_curves[i][j]['imu']['gyro'].values)
                angles_2['{}{}'.format(i,j)]= resample_fixed(pd.DataFrame(-sf['eu'][:,1]), 100).values

In [83]:
import numpy as np
import matplotlib.pyplot as plt
from tslearn.clustering import TimeSeriesKMeans
from tslearn.utils import to_time_series_dataset

X = np.array([
    angles_2['11'], angles_2['12'], angles_2['13'],
    angles_2['21'], angles_2['22'], angles_2['23'], angles_2['24'], angles_2['25'], 
    angles_2['31'], angles_2['32'], angles_2['33'], angles_2['34'], angles_2['35'], 
    angles_2['41'], angles_2['42'], angles_2['43'], angles_2['44'], angles_2['45'], 
    angles_2['51'], angles_2['52'], angles_2['53'], angles_2['54'],
    angles_2['61'], angles_2['62'], angles_2['63'], angles_2['64'], angles_2['65'],
    angles_2['71'], angles_2['72'], angles_2['73'], angles_2['74'], angles_2['75'], 
    angles_2['81'], angles_2['82'], angles_2['83'], angles_2['84'], angles_2['85'], 
    angles_2['91'], angles_2['92'], angles_2['93'], angles_2['94'], angles_2['95'], 
    angles_2['101'], angles_2['102'], angles_2['103'], angles_2['104'], 
    angles_2['111'], angles_2['112'], angles_2['113'], angles_2['114'], angles_2['115'], 
    angles_2['121'], angles_2['122'], angles_2['123']
    ]) 

n_features = 1
# n_clusters = 3       
n_samples = len(X)

from kneed import KneeLocator  # Import the KneeLocator from kneed

# Range of clusters to try
cluster_range = range(1, 11)  # Try from 1 to 10 clusters
inertia_values = []
# Run TimeSeriesKMeans for different cluster sizes
for n_clusters in cluster_range:
    model_bi_imu = TimeSeriesKMeans(n_clusters=n_clusters,  n_init=5, random_state=seed, metric="euclidean", verbose=False)
    model_bi_imu.fit(X)
    inertia_values.append(model_bi_imu.inertia_)

# Use KneeLocator to find the optimal "knee" (elbow point) in the plot
kneedle = KneeLocator(cluster_range, inertia_values, curve='convex', direction='decreasing')

# Get the "knee" point, i.e., the optimal number of clusters
n_clusters = kneedle.elbow
print(f"Optimal number of clusters: {n_clusters}")

from tslearn.preprocessing import TimeSeriesScalerMeanVariance, TimeSeriesResampler

seed = 0
np.random.seed(seed)
n_clusters = 2
# X = TimeSeriesScalerMeanVariance().fit_transform(X)
# X_train = TimeSeriesResampler(sz=40).fit_transform(X_train) # Make time series shorter
model_bi_imu = TimeSeriesKMeans(n_clusters=n_clusters, n_init = 5, random_state=seed).fit(X)
y_pred_2_imu = model_bi_imu.fit_predict(X)

plt.style.use('default')
# Step 3: Plot the cluster centers (these represent the "average" time-series for each cluster)
f, axs = plt.subplots(1,2,figsize=(20, 6))
f.suptitle('Bimanual lifting', fontsize=20)

colors = ['m', 'g']
# Plot each cluster center (the mean time-series in each cluster)
for cluster_idx, lab, c in zip([1,0], ['Hybrid', 'Squat'], colors):
    axs[0].plot(-model_bi_imu.cluster_centers_[cluster_idx][:, 0], label=f" {lab} - L5", color=c, linestyle='solid')  # Feature 1

# axs[0].set_title("Cluster Centers with TimeSeriesKMeans (Multivariate Time Series)")
axs[0].set_title("Cluster Centers", size=14)
axs[0].set_xlabel("Movement cycle (%)", size=14)
axs[0].set_ylabel("Flexion angles (deg)", size=14)
axs[0].legend(fontsize=14)

# Step 4: Print predicted cluster labels for the first few time-series
print("Predicted cluster labels for all time-series:", y_pred_2_imu)
# print(pd.DataFrame({'subj':subj, 'cluster':y_pred_2_imu}))

LABEL_COLOR_MAP = { 0 : 'g',
                    1 : 'm',
                   }

label_color = [LABEL_COLOR_MAP[l] for l in y_pred_2_imu]


# Optionally, visualize how the time-series are grouped into clusters
labeled_colors = set()

for i, c in zip(range(n_samples), label_color):
    if c not in labeled_colors:
        if c == 'm': lb = 'Hybrid'
        else: lb = 'Squat'

        axs[1].plot(-X[i, :, 0], color=label_color[i], label=lb)
        labeled_colors.add(c)
    else:
        axs[1].plot(-X[i, :, 0], color=label_color[i])

axs[1].set_title("L5 flexion angles grouped by strategy", fontsize=14)
# axs[1].set_ylabel("Degrees")
axs[0].set_ylim([-10, 80])
axs[1].set_ylim([-10, 80])
axs[1].legend(fontsize=14)

axs[1].set_xlabel("Movement cycle (%)", size=14)
axs[1].set_ylabel("L5 flexion angles (deg)", size=14)
plt.tight_layout(rect=[0, 0, 1, 0.99])  # reserve space for suptitle

pd.DataFrame({'subj_2':subj_2, 'results':y_pred_2_imu})

/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/pyth

Optimal number of clusters: 3


/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Predicted cluster labels for all time-series: [1 1 1 1 1 1 1 1 0 0 0 0 0 1 1 1 1 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 0 1 1 1 0 0 1 1 1]


,subj_2,results
0,1,1
1,1,1
2,1,1
3,2,1
4,2,1
5,2,1
6,2,1
7,2,1
8,3,0
9,3,0


2025-12-02 17:11:26.019 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:11:26.124 python[3160:47829] +[CATransaction synchronize] called within transaction


# Duration

In [ ]:
dd1, da1, dtot1 = [], [], []
for t in t1_curves:
    for tt in t1_curves[t]:
        if tt in ['points', 'whole_trunk', 'weight']: continue
        
        peaks, val = find_peaks(-t1_curves[t][tt]['com_vt']) 
        peak = peaks[0]

        dd1.append(len(t1_curves[t][tt]['com_vt'][:peak])/200)
        da1.append(len(t1_curves[t][tt]['com_vt'][peak:])/200)
        dtot1.append(len(t1_curves[t][tt]['com_vt'])/200)


dd2, da2, dtot2 = [], [], []
for t in t2_curves:
    for tt in t2_curves[t]:
        if tt in ['points', 'whole_trunk', 'weight']: continue
        
        peaks, val = find_peaks(-t2_curves[t][tt]['com_vt']) 
        peak = peaks[0]

        dd2.append(len(t2_curves[t][tt]['com_vt'][:peak])/200)
        da2.append(len(t2_curves[t][tt]['com_vt'][peak:])/200)
        dtot2.append(len(t2_curves[t][tt]['com_vt'])/200)


In [ ]:
dur1 = pd.DataFrame({'subj_1':subj_1, 'results':y_pred_1, 'dur_descent':dd1, 'dur_ascent':da1, 'dur_tot':dtot1})
dur1.groupby("results").mean(numeric_only=True)

In [ ]:
dur1.groupby("results").std(numeric_only=True)

In [ ]:
dur2 = pd.DataFrame({'subj_1':subj_2, 'results':y_pred_2, 'dur_descent':dd2, 'dur_ascent':da2, 'dur_tot':dtot2})
dur2.groupby("results").mean(numeric_only=True)

In [ ]:
dur2.groupby("results").std(numeric_only=True)

# Plot

## Biomechnical curves

In [60]:
from matplotlib.ticker import MaxNLocator

plt.style.use('default')
# plt.style.use('dark_background')
# uni
cl2 = ['11','12', '21', '22', '23', '24', '25', '31', '32', '33', '41', '42', '43', '44', '45', '61', '62', '63', '64', '71',
       '82', '111', '112', '113', '114', '115', '121']
cl1 = ['51', '52', '53','54', '65', '72', '73', '74', '75', '81', '83', '84', '85', '91', '92','93','94','95','101','102']


com_vt_registered_mean_cl1 = com_vt_registered_1[cl1].mean(numeric_only=True, axis=1)
com_vt_registered_std_cl1 = com_vt_registered_1[cl1].std(numeric_only=True, axis=1)
com_ap_registered_mean_cl1 = com_ap_registered_1[cl1].mean(numeric_only=True, axis=1)
com_ap_registered_std_cl1 = com_ap_registered_1[cl1].std(numeric_only=True, axis=1)
cop_ap_registered_mean_cl1 = cop_ap_registered_1[cl1].mean(numeric_only=True, axis=1)
cop_ap_registered_std_cl1 = cop_ap_registered_1[cl1].std(numeric_only=True, axis=1)
trunk_registered_mean_cl1 = (-trunk_registered_1[cl1]).mean(numeric_only=True, axis=1)
trunk_registered_std_cl1 = (-trunk_registered_1[cl1]).std(numeric_only=True, axis=1)
knee_registered_mean_cl1 = knee_registered_1[cl1].mean(numeric_only=True, axis=1)
knee_registered_std_cl1 = knee_registered_1[cl1].std(numeric_only=True, axis=1)

com_vt_registered_mean_cl2 = com_vt_registered_1[cl2].mean(numeric_only=True, axis=1)
com_vt_registered_std_cl2 = com_vt_registered_1[cl2].std(numeric_only=True, axis=1)
com_ap_registered_mean_cl2 = com_ap_registered_1[cl2].mean(numeric_only=True, axis=1)
com_ap_registered_std_cl2 = com_ap_registered_1[cl2].std(numeric_only=True, axis=1)
cop_ap_registered_mean_cl2 = cop_ap_registered_1[cl2].mean(numeric_only=True, axis=1)
cop_ap_registered_std_cl2 = cop_ap_registered_1[cl2].std(numeric_only=True, axis=1)
trunk_registered_mean_cl2 = (-trunk_registered_1[cl2]).mean(numeric_only=True, axis=1)
trunk_registered_std_cl2 = (-trunk_registered_1[cl2]).std(numeric_only=True, axis=1)
knee_registered_mean_cl2 = knee_registered_1[cl2].mean(numeric_only=True, axis=1)
knee_registered_std_cl2 = knee_registered_1[cl2].std(numeric_only=True, axis=1)

bos_1_mean_cl1 = bos_1[cl1].mean(numeric_only=True, axis=1)
bos_1_std_cl1 = bos_1[cl1].std(numeric_only=True, axis=1)
bos_1_mean_cl2 = bos_1[cl2].mean(numeric_only=True, axis=1)
bos_1_std_cl2 = bos_1[cl2].std(numeric_only=True, axis=1)

fz_registered_mean_cl1 = fz_1[cl1].mean(numeric_only=True, axis=1)
fz_registered_std_cl1 = fz_1[cl1].std(numeric_only=True, axis=1)
fz_registered_mean_cl2 = fz_1[cl2].mean(numeric_only=True, axis=1)
fz_registered_std_cl2 = fz_1[cl2].std(numeric_only=True, axis=1)
fy_registered_mean_cl1 = fy_1[cl1].mean(numeric_only=True, axis=1)
fy_registered_std_cl1 = fy_1[cl1].std(numeric_only=True, axis=1)
fy_registered_mean_cl2 = fy_1[cl2].mean(numeric_only=True, axis=1)
fy_registered_std_cl2 = fy_1[cl2].std(numeric_only=True, axis=1)

f, axs = plt.subplots(1, 6, figsize=[20,4], sharey='col')
f.suptitle('Unimanual lifting', fontsize=20)
x = np.linspace(0, 100, 100)
axs[0].plot(com_vt_registered_mean_cl2, color='m', label='Hybrid', linewidth=2)
axs[0].fill_between(x, (com_vt_registered_mean_cl2-com_vt_registered_std_cl2),(com_vt_registered_mean_cl2+com_vt_registered_std_cl2), color='m', alpha=.4)
axs[0].plot(com_vt_registered_mean_cl1, color='g', label='Squat', linewidth=2)
axs[0].fill_between(x, (com_vt_registered_mean_cl1-com_vt_registered_std_cl1),(com_vt_registered_mean_cl1+com_vt_registered_std_cl1), color='g', alpha=.4)

axs[1].plot(com_ap_registered_mean_cl1, color='g', label='mean', linewidth=2)
axs[1].fill_between(x, (com_ap_registered_mean_cl1-com_ap_registered_std_cl1),(com_ap_registered_mean_cl1+com_ap_registered_std_cl1), color='g', label='std', alpha=.4)
axs[1].plot(com_ap_registered_mean_cl2, color='m', label='mean', linewidth=2)
axs[1].fill_between(x, (com_ap_registered_mean_cl2-com_ap_registered_std_cl2),(com_ap_registered_mean_cl2+com_ap_registered_std_cl2), color='m', alpha=.4)

axs[2].plot(cop_ap_registered_mean_cl1, color='g', label='mean', linewidth=2)
axs[2].fill_between(x, (cop_ap_registered_mean_cl1-cop_ap_registered_std_cl1),(cop_ap_registered_mean_cl1+cop_ap_registered_std_cl1), color='g', label='std', alpha=.4)
axs[2].plot(cop_ap_registered_mean_cl2, color='m', label='mean', linewidth=2)
axs[2].fill_between(x, (cop_ap_registered_mean_cl2-cop_ap_registered_std_cl2),(cop_ap_registered_mean_cl2+cop_ap_registered_std_cl2), color='m', alpha=.4)

axs[3].plot(bos_1_mean_cl1, color='g', label='mean', linewidth=2)
axs[3].fill_between(x, (bos_1_mean_cl1-bos_1_std_cl1),(bos_1_mean_cl1+bos_1_std_cl1), color='g', label='std', alpha=.4)
axs[3].plot(bos_1_mean_cl2, color='m', label='mean', linewidth=2)
axs[3].fill_between(x, (bos_1_mean_cl2-bos_1_std_cl2),(bos_1_mean_cl2+bos_1_std_cl2), color='m', alpha=.4)

axs[4].plot(fz_registered_mean_cl1, color='g', label='mean', linewidth=2)
axs[4].fill_between(x, (fz_registered_mean_cl1-fz_registered_std_cl1),(fz_registered_mean_cl1+fz_registered_std_cl1), color='g', label='std', alpha=.4)
axs[4].plot(fz_registered_mean_cl2, color='m', label='mean', linewidth=2)
axs[4].fill_between(x, (fz_registered_mean_cl2-fz_registered_std_cl2),(fz_registered_mean_cl2+fz_registered_std_cl2), color='m', alpha=.4)

axs[5].plot(fy_registered_mean_cl1, color='g', label='mean', linewidth=2)
axs[5].fill_between(x, (fy_registered_mean_cl1-fy_registered_std_cl1),(fy_registered_mean_cl1+fy_registered_std_cl1), color='g', label='std', alpha=.4)
axs[5].plot(fy_registered_mean_cl2, color='m', label='mean', linewidth=2)
axs[5].fill_between(x, (fy_registered_mean_cl2-fy_registered_std_cl2),(fy_registered_mean_cl2+fy_registered_std_cl2), color='m', alpha=.4)

axs[0].set_ylabel('Norm COM-VT',size=14)
axs[1].set_ylabel('Norm COM-AP',size=14)
axs[2].set_ylabel('Norm COP-AP',size=14)
axs[3].set_ylabel('Norm BOS [$m$]',size=14)
axs[4].set_ylabel('Norm GRF-VT [N/Kg]',size=14)
axs[5].set_ylabel('Norm GRF-AP [N/Kg]',size=14)
axs[0].legend(loc='upper center',fontsize=14)
for i in range(6):
       axs[i].set_xlabel('Movement cycle (%)',size=14)
       axs[i].yaxis.set_major_locator(MaxNLocator(nbins=5))  # same # of ticks

# =========
# bi
cl2 = ['11','12', '13','21', '22', '23', '24', '25', '41', '42', '43', '44', '45', '61', '62', '63', '64', '65',
       '82', '111', '112', '113', '114', '115', '121', '122', '123']
cl1 = [ '31', '32', '33', '34', '35','51', '52', '53', '54', '71','72', '73', '74', '75', '81', '83', '84', '85',
          '91', '92','93','94','95','101','102', '103', '104']

com_vt_registered_mean_cl1 = com_vt_registered_2[cl1].mean(numeric_only=True, axis=1)
com_vt_registered_std_cl1 = com_vt_registered_2[cl1].std(numeric_only=True, axis=1)
com_ap_registered_mean_cl1 = com_ap_registered_2[cl1].mean(numeric_only=True, axis=1)
com_ap_registered_std_cl1 = com_ap_registered_2[cl1].std(numeric_only=True, axis=1)
cop_ap_registered_mean_cl1 = cop_ap_registered_2[cl1].mean(numeric_only=True, axis=1)
cop_ap_registered_std_cl1 = cop_ap_registered_2[cl1].std(numeric_only=True, axis=1)
trunk_registered_mean_cl1 = (-trunk_registered_2[cl1]).mean(numeric_only=True, axis=1)
trunk_registered_std_cl1 = (-trunk_registered_2[cl1]).std(numeric_only=True, axis=1)
knee_registered_mean_cl1 = knee_registered_2[cl1].mean(numeric_only=True, axis=1)
knee_registered_std_cl1 = knee_registered_2[cl1].std(numeric_only=True, axis=1)

com_vt_registered_mean_cl2 = com_vt_registered_2[cl2].mean(numeric_only=True, axis=1)
com_vt_registered_std_cl2 = com_vt_registered_2[cl2].std(numeric_only=True, axis=1)
com_ap_registered_mean_cl2 = com_ap_registered_2[cl2].mean(numeric_only=True, axis=1)
com_ap_registered_std_cl2 = com_ap_registered_2[cl2].std(numeric_only=True, axis=1)
cop_ap_registered_mean_cl2 = cop_ap_registered_2[cl2].mean(numeric_only=True, axis=1)
cop_ap_registered_std_cl2 = cop_ap_registered_2[cl2].std(numeric_only=True, axis=1)
trunk_registered_mean_cl2 = (-trunk_registered_2[cl2]).mean(numeric_only=True, axis=1)
trunk_registered_std_cl2 = (-trunk_registered_2[cl2]).std(numeric_only=True, axis=1)
knee_registered_mean_cl2 = knee_registered_2[cl2].mean(numeric_only=True, axis=1)
knee_registered_std_cl2 = knee_registered_2[cl2].std(numeric_only=True, axis=1)

bos_2_mean_cl1 = bos_2[cl1].mean(numeric_only=True, axis=1)
bos_2_std_cl1 = bos_2[cl1].std(numeric_only=True, axis=1)
bos_2_mean_cl2 = bos_2[cl2].mean(numeric_only=True, axis=1)
bos_2_std_cl2 = bos_2[cl2].std(numeric_only=True, axis=1)

fz_registered_mean_cl1 = fz_2[cl1].mean(numeric_only=True, axis=1)
fz_registered_std_cl1 = fz_2[cl1].std(numeric_only=True, axis=1)
fz_registered_mean_cl2 = fz_2[cl2].mean(numeric_only=True, axis=1)
fz_registered_std_cl2 = fz_2[cl2].std(numeric_only=True, axis=1)
fy_registered_mean_cl1 = fy_2[cl1].mean(numeric_only=True, axis=1)
fy_registered_std_cl1 = fy_2[cl1].std(numeric_only=True, axis=1)
fy_registered_mean_cl2 = fy_2[cl2].mean(numeric_only=True, axis=1)
fy_registered_std_cl2 = fy_2[cl2].std(numeric_only=True, axis=1)
plt.tight_layout()

f, axs = plt.subplots(1, 6, figsize=[20,4], sharey='col')
f.suptitle('Bimanual lifting', fontsize=20)

x = np.linspace(0, 100, 100)
axs[0].plot(com_vt_registered_mean_cl2, color='m', label='Hybrid', linewidth=2)
axs[0].fill_between(x, (com_vt_registered_mean_cl2-com_vt_registered_std_cl2),(com_vt_registered_mean_cl2+com_vt_registered_std_cl2), color='m', alpha=.4)
axs[0].plot(com_vt_registered_mean_cl1, color='g', label='Squat', linewidth=2)
axs[0].fill_between(x, (com_vt_registered_mean_cl1-com_vt_registered_std_cl1),(com_vt_registered_mean_cl1+com_vt_registered_std_cl1), color='g', alpha=.4)

axs[1].plot(com_ap_registered_mean_cl1, color='g', label='mean', linewidth=2)
axs[1].fill_between(x, (com_ap_registered_mean_cl1-com_ap_registered_std_cl1),(com_ap_registered_mean_cl1+com_ap_registered_std_cl1), color='g', label='std', alpha=.4)
axs[1].plot(com_ap_registered_mean_cl2, color='m', label='mean', linewidth=2)
axs[1].fill_between(x, (com_ap_registered_mean_cl2-com_ap_registered_std_cl2),(com_ap_registered_mean_cl2+com_ap_registered_std_cl2), color='m', alpha=.4)

axs[2].plot(cop_ap_registered_mean_cl1, color='g', label='mean', linewidth=2)
axs[2].fill_between(x, (cop_ap_registered_mean_cl1-cop_ap_registered_std_cl1),(cop_ap_registered_mean_cl1+cop_ap_registered_std_cl1), color='g', label='std', alpha=.4)
axs[2].plot(cop_ap_registered_mean_cl2, color='m', label='mean', linewidth=2)
axs[2].fill_between(x, (cop_ap_registered_mean_cl2-cop_ap_registered_std_cl2),(cop_ap_registered_mean_cl2+cop_ap_registered_std_cl2), color='m', alpha=.4)

axs[3].plot(bos_2_mean_cl1, color='g', label='mean', linewidth=2)
axs[3].fill_between(x, (bos_2_mean_cl1-bos_2_std_cl1),(bos_2_mean_cl1+bos_2_std_cl1), color='g', label='std', alpha=.4)
axs[3].plot(bos_2_mean_cl2, color='m', label='mean', linewidth=2)
axs[3].fill_between(x, (bos_2_mean_cl2-bos_2_std_cl2),(bos_2_mean_cl2+bos_2_std_cl2), color='m', alpha=.4)

axs[4].plot(fz_registered_mean_cl1, color='g', label='mean', linewidth=2)
axs[4].fill_between(x, (fz_registered_mean_cl1-fz_registered_std_cl1),(fz_registered_mean_cl1+fz_registered_std_cl1), color='g', label='std', alpha=.4)
axs[4].plot(fz_registered_mean_cl2, color='m', label='mean', linewidth=2)
axs[4].fill_between(x, (fz_registered_mean_cl2-fz_registered_std_cl2),(fz_registered_mean_cl2+fz_registered_std_cl2), color='m', alpha=.4)

axs[5].plot(fy_registered_mean_cl1, color='g', label='mean', linewidth=2)
axs[5].fill_between(x, (fy_registered_mean_cl1-fy_registered_std_cl1),(fy_registered_mean_cl1+fy_registered_std_cl1), color='g', label='std', alpha=.4)
axs[5].plot(fy_registered_mean_cl2, color='m', label='mean', linewidth=2)
axs[5].fill_between(x, (fy_registered_mean_cl2-fy_registered_std_cl2),(fy_registered_mean_cl2+fy_registered_std_cl2), color='m', alpha=.4)

axs[0].set_ylabel('Norm COM-VT',size=14)
axs[1].set_ylabel('Norm COM-AP',size=14)
axs[2].set_ylabel('Norm COP-AP',size=14)
axs[3].set_ylabel('Norm BOS [$m$]',size=14)
axs[4].set_ylabel('Norm GRF-VT [N/Kg]',size=14)
axs[5].set_ylabel('Norm GRF-AP [N/Kg]',size=14)
axs[0].legend(loc='upper center',fontsize=14)
for i in range(6):
       axs[i].set_xlabel('Movement cycle (%)',size=14)
       axs[i].yaxis.set_major_locator(MaxNLocator(nbins=5))  # same # of ticks

# plt.tight_layout(pad=2)


# axs[0].set_title('VT COM displacement', fontsize=15)
# axs[1].set_title('AP COM displacement', fontsize=15)
# axs[2].set_title('AP COP displacement', fontsize=15)
# axs[3].set_title('Base Of Support', fontsize=15)
# axs[4].set_title('VT GRF', fontsize=15)
# axs[5].set_title('AP GRF', fontsize=15)

plt.tight_layout()

2025-12-02 15:04:39.130 python[29629:1628888] +[CATransaction synchronize] called within transaction
2025-12-02 15:04:39.285 python[29629:1628888] +[CATransaction synchronize] called within transaction
2025-12-02 15:04:47.449 python[29629:1628888] +[CATransaction synchronize] called within transaction
2025-12-02 15:04:47.612 python[29629:1628888] +[CATransaction synchronize] called within transaction
2025-12-02 15:04:53.016 python[29629:1628888] +[CATransaction synchronize] called within transaction
2025-12-02 15:04:53.758 python[29629:1628888] +[CATransaction synchronize] called within transaction
2025-12-02 15:04:54.031 python[29629:1628888] +[CATransaction synchronize] called within transaction


# SPM

## per subject

In [30]:
fmag_1 = {}
for f in fx_1:
    fmag_1['{}'.format(f)] = (fx_1['{}'.format(f)]**2+fy_1['{}'.format(f)]**2+fz_1['{}'.format(f)]**2)**(1/2)

fmag_1 = pd.DataFrame(fmag_1)

fmag_2 = {}
for f in fx_2:
    fmag_2['{}'.format(f)] = (fx_2['{}'.format(f)]**2+fy_2['{}'.format(f)]**2+fz_2['{}'.format(f)]**2)**(1/2)

fmag_2 = pd.DataFrame(fmag_2)


b, a = signal.butter(4, 50, 'lowpass', fs=200)
fmag_1_filt = {}
for f in fmag_1:
    fmag_1_filt['{}'.format(f)] = signal.filtfilt(b, a, fmag_1['{}'.format(f)])

fmag_2_filt = {}
for f in fmag_2:
    fmag_2_filt['{}'.format(f)] = signal.filtfilt(b, a, fmag_2['{}'.format(f)])

fmag_1_filt = pd.DataFrame(fmag_1_filt)
fmag_2_filt = pd.DataFrame(fmag_2_filt)

fx_1_filt, fy_1_filt, fz_1_filt = {}, {}, {}
for f in fx_1:
    fx_1_filt['{}'.format(f)] = signal.filtfilt(b, a, fx_1['{}'.format(f)])
    fy_1_filt['{}'.format(f)] = signal.filtfilt(b, a, fy_1['{}'.format(f)])
    fz_1_filt['{}'.format(f)] = signal.filtfilt(b, a, fz_1['{}'.format(f)])

fx_2_filt, fy_2_filt, fz_2_filt = {}, {}, {}
for f in fx_2:
    fx_2_filt['{}'.format(f)] = signal.filtfilt(b, a, fx_2['{}'.format(f)])
    fy_2_filt['{}'.format(f)] = signal.filtfilt(b, a, fy_2['{}'.format(f)])
    fz_2_filt['{}'.format(f)] = signal.filtfilt(b, a, fz_2['{}'.format(f)])

fx_1_filt = pd.DataFrame(fx_1_filt)
fy_1_filt = pd.DataFrame(fy_1_filt)
fz_1_filt = pd.DataFrame(fz_1_filt)
fx_2_filt = pd.DataFrame(fx_2_filt)
fy_2_filt = pd.DataFrame(fy_2_filt)
fz_2_filt = pd.DataFrame(fz_2_filt)
# Accorpo i soggetti
fmag_acc_1,strat_1 = {}, [1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1]
fmag_acc_1['1'] = fmag_1_filt[['11', '12']].mean(axis=1)
fmag_acc_1['2'] = fmag_1_filt[['21', '22', '23', '24', '25']].mean(axis=1)
fmag_acc_1['3'] = fmag_1_filt[['31', '32', '33']].mean(axis=1)
fmag_acc_1['4'] = fmag_1_filt[['41', '42', '43', '44', '45']].mean(axis=1)
fmag_acc_1['5'] = fmag_1_filt[['51', '52', '53', '54', '55']].mean(axis=1)
fmag_acc_1['6'] = fmag_1_filt[['61', '62', '63', '64']].mean(axis=1)
fmag_acc_1['7'] = fmag_1_filt[['71', '72', '73', '74', '75']].mean(axis=1)
fmag_acc_1['8'] = fmag_1_filt[['81', '82', '83', '84', '85']].mean(axis=1)
fmag_acc_1['9'] = fmag_1_filt[['91', '92', '93', '94', '95']].mean(axis=1)
fmag_acc_1['10'] = fmag_1_filt[['101', '102']].mean(axis=1)
fmag_acc_1['11'] = fmag_1_filt[['111', '112', '113', '114', '115']].mean(axis=1)
fmag_acc_1['12'] = fmag_1_filt[['121']].mean(axis=1)
fmag_acc_1 = pd.DataFrame(fmag_acc_1)

fmag_acc_2, strat_2 = {}, [1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1]
fmag_acc_2['1']= fmag_2_filt[['11', '12', '13']].mean(axis=1)
fmag_acc_2['2']= fmag_2_filt[['21', '22', '23', '24', '25']].mean(axis=1)
fmag_acc_2['3']= fmag_2_filt[['31', '32', '33', '34', '35']].mean(axis=1)
fmag_acc_2['4']= fmag_2_filt[['41', '42', '43', '44', '45']].mean(axis=1)
fmag_acc_2['5']= fmag_2_filt[['51', '52', '53', '54']].mean(axis=1)
fmag_acc_2['6']= fmag_2_filt[['61', '62', '63', '64']].mean(axis=1)
fmag_acc_2['7']= fmag_2_filt[['72', '73', '74', '75']].mean(axis=1)
fmag_acc_2['8']= fmag_2_filt[['81', '82', '83', '84', '85']].mean(axis=1)
fmag_acc_2['9']= fmag_2_filt[['91', '92', '93', '94', '95']].mean(axis=1)
fmag_acc_2['10'] = fmag_2_filt[['101', '102', '103', '104']].mean(axis=1)
fmag_acc_2['11'] = fmag_2_filt[['111', '112', '113', '114', '115']].mean(axis=1)
fmag_acc_2['12'] = fmag_2_filt[['121', '122', '123']].mean(axis=1)
fmag_acc_2 = pd.DataFrame(fmag_acc_2)

In [51]:
alpha        = 0.05


# Unimanual
A_1 = np.array([1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1])
Y_1 = np.array(fmag_acc_1[1:]).transpose()

#(1) Run ANOVA:
F_1  = spm1d.stats.anova1(Y_1, A_1, equal_var=False)
Fi_1 = F_1.inference(alpha, interp=True, circular=False)
print( Fi_1 )

#(2) Plot results:
plt.figure(figsize=[10,5])
plt.subplot(1,2,1)
Fi_1.plot()
Fi_1.plot_threshold_label(bbox=dict(facecolor='w'))
Fi_1.plot_p_values()
plt.ylim(0, 50)
# plt.xlabel('(%) descent-ascent time', size=10)
plt.title('Unimanual lifting', fontsize=20)


# Bimanual
A_2 = np.array([1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1])
Y_2 = np.array(fmag_acc_2[1:]).transpose()
#(1) Run ANOVA:
F_2 = spm1d.stats.anova1(Y_2, A_2, equal_var=False)
Fi_2 = F_2.inference(alpha, interp=True, circular=False)
print( Fi_2 )

#(2) Plot results:
plt.subplot(1,2,2)
Fi_2.plot()
Fi_2.plot_threshold_label(bbox=dict(facecolor='w'))
Fi_2.plot_p_values()
plt.ylim(0, 50)
# plt.xlabel('(%) descent-ascent time', size=10)
plt.title('Bimanual lifting', fontsize=20)

plt.show()


SPM{F} inference field
   SPM.effect    :   Main A
   SPM.z         :  (1x99) raw test stat field
   SPM.df        :  (1.713, 8.396)
   SPM.fwhm      :  8.52362
   SPM.resels    :  (1, 11.49746)
Inference:
   SPM.alpha     :  0.050
   SPM.zstar     :  17.80714
   SPM.h0reject  :  True
   SPM.p_set     :  <0.001
   SPM.p_cluster :  (<0.001, 0.016)



SPM{F} inference field
   SPM.effect    :   Main A
   SPM.z         :  (1x99) raw test stat field
   SPM.df        :  (1.832, 9.093)
   SPM.fwhm      :  8.51172
   SPM.resels    :  (1, 11.51353)
Inference:
   SPM.alpha     :  0.050
   SPM.zstar     :  15.90336
   SPM.h0reject  :  True
   SPM.p_set     :  0.001
   SPM.p_cluster :  (<0.001, 0.044)





/Users/paoladiflorio/opt/anaconda3/envs/dataManager/lib/python3.9/site-packages/spm1d/stats/_datachecks.py:204: UserWarning: 

  checker.check()
/var/folders/9s/6wtf_gh12cv5t_tts750b9jh0000gn/T/ipykernel_3160/2550176531.py:9: UserWarning: 

  F_1  = spm1d.stats.anova1(Y_1, A_1, equal_var=False)
/var/folders/9s/6wtf_gh12cv5t_tts750b9jh0000gn/T/ipykernel_3160/2550176531.py:28: UserWarning: 

  F_2 = spm1d.stats.anova1(Y_2, A_2, equal_var=False)


In [ ]:
# Unimanual
# Fi is the SPM inference object
F_values_1 = Fi_1.z       # the F-statistic curve
F_thresh_1 = Fi_1.zstar   # critical threshold for significance
p_values_1 = Fi_1.p       # pointwise p-values
sig_mask_1 = F_values_1 > F_thresh_1    # boolean array where the F-value is significant
time_1 = np.linspace(0, 100, len(F_values_1))  # x-axis (% movement cycle)

# Bimanual
# Fi is the SPM inference object
F_values_2 = Fi_2.z       # the F-statistic curve
F_thresh_2 = Fi_2.zstar   # critical threshold for significance
p_values_2 = Fi_2.p       # pointwise p-values
sig_mask_2 = F_values_2 > F_thresh_2    # boolean array where the F-value is significant
time_2 = np.linspace(0, 100, len(F_values_2))  # x-axis (% movement cycle)

# Plot

f, axs = plt.subplots(1,2,figsize=[15,5],sharey=True)

axs[0].plot(time_1, F_values_1, color='k')
axs[0].axhline(F_thresh_1, color='r', linestyle='--', label=f'F-threshold={np.round(F_thresh_1,3)}')
axs[0].fill_between(time_1, 0, F_values_1, where=sig_mask_1, color='grey', alpha=0.3, label='p<0.01')
# plt.xlabel('Movement cycle (%)')
axs[0].set_ylabel('F-value', size=14)
axs[0].set_title('Unimanual lifting', fontsize=20)
axs[0].legend(fontsize=14)

axs[1].plot(time_2, F_values_2, color='k')
axs[1].axhline(F_thresh_2, color='r', linestyle='--', label=f'F-threshold={np.round(F_thresh_2,3)}')
axs[1].fill_between(time_2, 0, F_values_2, where=sig_mask_2, color='grey', alpha=0.3, label='p<0.01')
# plt.xlabel('Movement cycle (%)')
axs[1].set_ylabel('F-value', size=14)
axs[1].set_title('Bimanual lifting', fontsize=20)
axs[1].legend(fontsize=14)

plt.show()


2025-12-02 17:02:45.834 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:02:45.955 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:02:49.721 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:02:50.872 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:02:51.139 python[3160:47829] +[CATransaction synchronize] called within transaction


# Confusion matrix

In [87]:
def matrice_confusione(y_true, y_pred):

    # Matrice di confusione
    cm = confusion_matrix(y_true, y_pred)
    print("Matrice di Confusione:")
    print(cm)

    # Calcolo della sensibilità (recall) e specificità
    TP = cm[1, 1]
    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]

    sensibilità = TP / (TP + FN)
    specificità = TN / (TN + FP)

    print(f"Sensibilità: {sensibilità:.2f}")
    print(f"Specificità: {specificità:.2f}")

    # ROC AUC
    roc_auc = roc_auc_score(y_true, y_pred)
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)

    # Grafico della curva ROC
    plt.figure(figsize=[5, 10])
    plt.subplot(2,1,1)
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive (FPR)')
    plt.ylabel('True Positive (TPR)')
    plt.title('ROC curve')
    plt.legend(loc='lower right')
    plt.show()

    plt.subplot(2,1,2)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Non Squat', 'Squat'], yticklabels=['Non Squat', 'Squat'])
    plt.xlabel('Pred')
    plt.ylabel('True')
    plt.title('Confusion matrix')

    plt.tight_layout(pad=2, w_pad=0.2, h_pad=2.0)
    plt.show()
    

    # AUC
    print(f"AUC: {roc_auc:.2f}")

uni = pd.DataFrame({'subj':subj_1, 'true':y_pred_1, 'pred':y_pred_1_imu})
bi = pd.DataFrame({'subj':subj_2, 'true':y_pred_2, 'pred':y_pred_2_imu})

cm_uni = matrice_confusione(uni['true'], uni['pred'])
cm_bi = matrice_confusione(bi['true'], bi['pred'])

Matrice di Confusione:
[[20  1]
 [ 8 19]]
Sensibilità: 0.70
Specificità: 0.95
AUC: 0.83
Matrice di Confusione:
[[27  0]
 [ 5 22]]
Sensibilità: 0.81
Specificità: 1.00
AUC: 0.91


In [91]:
f, ax = plt.subplots(1, 2, figsize=[10, 5])
f.suptitle('Confusion matrix', fontsize=20)
# --- Unimanual lifting ---
cm_uni = confusion_matrix(uni['true'], uni['pred'])
TP = cm_uni[1, 1]
TN = cm_uni[0, 0]
FP = cm_uni[0, 1]
FN = cm_uni[1, 0]
sensibilità = TP / (TP + FN)
specificità = TN / (TN + FP)
print("Unimanual lifting Confusion Matrix:\n", cm_uni)

sns.heatmap(
    cm_uni, annot=True, fmt='d', cmap='Blues', cbar=False,
    xticklabels=['Non Squat', 'Squat'], yticklabels=['Non Squat', 'Squat'],
    ax=ax[0]  # <- specify the axis
)
ax[0].set_xlabel('Pred',size=14)
ax[0].set_ylabel('True',size=14)
ax[0].set_title('Unimanual lifting', fontsize=14)

# --- Bimanual lifting ---
cm_bi = confusion_matrix(bi['true'], bi['pred'])
TP = cm_bi[1, 1]
TN = cm_bi[0, 0]
FP = cm_bi[0, 1]
FN = cm_bi[1, 0]
sensibilità = TP / (TP + FN)
specificità = TN / (TN + FP)
print("Bimanual lifting Confusion Matrix:\n", cm_bi)

sns.heatmap(
    cm_bi, annot=True, fmt='d', cmap='Blues', cbar=False,
    xticklabels=['Non Squat', 'Squat'], yticklabels=['Non Squat', 'Squat'],
    ax=ax[1]  # <- specify the axis
)
ax[1].set_xlabel('Pred',size=14)
ax[1].set_ylabel('True',size=14)
ax[1].set_title('Bimanual lifting', fontsize=14)

plt.tight_layout()
plt.show()

Unimanual lifting Confusion Matrix:
 [[20  1]
 [ 8 19]]
Bimanual lifting Confusion Matrix:
 [[27  0]
 [ 5 22]]


2025-12-02 17:14:38.927 python[3160:47829] +[CATransaction synchronize] called within transaction
2025-12-02 17:14:39.047 python[3160:47829] +[CATransaction synchronize] called within transaction
